
# Real vs. AI-generated Image Detection

## Instructions

Use this notebook as the **official project report + code submission**.



### Rules
- Write explanations in **Markdown cells** and implementation in **Code cells**.
- The notebook must run **top to bottom**.
- Every claim about performance must be supported by results shown in the notebook.
- Clearly report any use of AI tools.



### Evaluation alignment
This notebook is structured to support evaluation on:
1. Problem definition and motivation
2. Baseline design
3. Model implementation
4. Evaluation methodology
5. Experimental results
6. Discussion and interpretation
7. Reproducibility



# Metadata

Please complete all fields below.

```yaml
project_title: "Real vs. AI-generated Image Detection"
dataset_name: "Labeled Deepfake Image Collection"
task_type: "Binary Image classification"
methods:
  - shallow_cnn_models
  - transfer_learning_resenet18_pretrained
  - random_search_hyperparameter_tuning
  - lime_explainability_analysis
  - integrated_gradients_explainability
  - reproducibility
evaluation_metrics:
  - accuracy
  - precision
  - recall
  - f1_score
  - roc_auc
  - confusion_matrix
team_members:
  - name: Vatsalya Arora
    student_id: 946827
    role: code_development, explanations, code_review
  - name: Ludovica Tapparo
    student_id: 880436
    role: code_development, explanations, project_parts_division_organization
  - name: Emily Jenks
    student_id: 947799
    role: code_development, explanations, code_review
  - name: Chiara Bosi
    student_id: 907066
    role: code_development, explanations, project_parts_division_organization


Submission date: 08-06-2026

```

# Contribution Tracking Rules

Each section MUST begin with:

```markdown
## Contribution Metadata
Author(s):  
Section:  
Method: [keyterms]
SHORT Description of contribution:

Difficulty: BASIC / INTERMEDIATE / ADVANCED
```


## AI Usage Declaration

Fill in the following table.

| Item | Response |
|---|---|
| Did you use generative AI tools? | Yes
| Which tools? | Claude, ChatGPT, Gemini
| For what purpose? | brainstorming, coding help, text editor
| Which parts were AI-assisted? | preprocessing methods, random search, model fine tuning for our objective
| What did you verify yourself? | We verified the entire pipeline

### Short declaration
We used AI to brainstorm an adequate project idea and corresponding pipeline, which would have been coherent and logical in its choices and steps. We also used it to understand advanced practical concepts of machine learning, and how to implement them. Finally, it was used to improve our explanation cells, making them clearer and less redundant.

## 1. Problem Definition and Motivation

```markdown
## Contribution Metadata
Author(s): Ludovica Tapparo
Section: Preprocessing part 1
Method: [problem_definition, structure_explanation]
SHORT Description of contribution:
In this section, I contributed by defining the real-vs-AI image classification
problem and explaining why deepfake detection is an important computer vision task.
I motivated the project by linking the social relevance of detecting synthetic
images to our technical goal of comparing a shallow CNN with a deeper ResNet18
architecture.

Difficulty: BASIC
```
### Deepfake Image Classification Problem

In recent years, the rapid development of generative artificial intelligence has made it increasingly easy to create highly realistic synthetic images. Tools based on deep learning can now generate faces, objects, scenes, and visual content that have grown to be more difficult to distinguish from real photographs. While these technologies could have useful applications in creativity, design, entertainment, and data generation, they also raise important concerns when synthetic images are used without disclosure. AI-generated images and deepfakes can be exploited to spread misinformation, manipulate public opinion, impersonate individuals, damage reputations, or reduce trust in digital media. As a result, the ability to automatically distinguish between real and AI-generated images has become an important task in computer vision and digital forensics.

Therefore, the problem we wish to address in this project is a binary image classification task: given an input image, the goal is to classify it as either real or AI-generated. This is a challenging problem because modern generative models are capable of producing images with realistic textures, lighting, colors, and facial or object structures. Therefore, the classifier cannot rely only on simple visual differences that are obvious to humans, but must learn more subtle patterns and artifacts that may be present in synthetic images. These may include inconsistencies in texture, unnatural local details, irregular edges, or statistical patterns introduced by the image generation process.

The motivation behind this project is to investigate how deep learning models can be used to solve this problem, tackling it with both a shallow method and a transfer learning method, in order to compare the two and investigate whether increasing model depth improves classification performance. For this reason, throughout this project we compare a shallow Convolutional Neural Network with a deeper architecture based on ResNet18. The shallow CNN serves as a simpler baseline model, allowing us to observe how a relatively small network performs when learning features directly from the dataset. ResNet18, instead, is used as a stronger proposed model because its deeper residual architecture can extract more complex and hierarchical image representations. By comparing these models, the project aims not only to build a classifier for real versus AI-generated images, but also to evaluate whether the improvement obtained by increasing the number of layers justifies the use of a deeper transfer learning model for this task.



## 2. Dataset Creation and Description

```markdown
## Contribution Metadata
Author(s): Vatsalya Arora
Section: Preprocessing part 2
Method: [DataDescription, ImportingDataset, Dataframe]
SHORT Description of contribution:
In this section, I imported the dataset in form of
a zip file by using a google drive link. Each file in the dataset is either real or fake, as denoted in their filename, for example "REAL_0001.jpg".
Therefore, the labels are mentioned in the filename. I created a dataframe, whose columns are filename,
binary label (0 for fake, 1 for real) and generator name.
Difficulty: BASIC
```
### Our Dataset
The dataset used in this project is the “Labeled Deepfake Image Collection”, available on Kaggle:
https://www.kaggle.com/datasets/jayanthbottu/labeled-deepfake-image-collection

The dataset contains 18,124 of images, including both real images (5,890) and deepfake images (12,234). Each image represents one data sample.

The dataset is separated into folders containing images. Due to the large scope of the data size, the following folders were selectively chosen.
- "Real" folder included 5,890 real images
- "DALL-E" folder included 2,000 AI generated images
- "DeepFaceLab" folder included 1,606 AI generated images
- "Midjourney" folder included 930 AI generated images

DALL-E, DeepFaceLab and Midjourney folders in this dataset were selected because they exhibit a decent level of visual realism, making them somewhat difficult to distinguish from real images with the naked eye.

###Data Import

In order to ensure a clean setup, the dataset is imported first by removing any existing image directory "Data". The data is then downloaded from Google Drive as a compressed ZIP file using gdown, extracted into a designated folder, and organized for access. After extraction, the total number of images (10,426) is verified by counting the files in the main directory. This process ensures that all images are properly loaded and ready for preprocessing and model training.


In [ ]:
import json
from PIL import Image, UnidentifiedImageError
from io import BytesIO
import gdown
import os
import requests
from google.colab import drive, files
import pandas as pd
import shutil
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
import torch
!pip install torchinfo
from torchinfo import summary
import torch.nn as nn
import itertools, random
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay
)
SEED= 42
import sys
import subprocess
import importlib.util
from skimage.segmentation import slic, mark_boundaries
!pip install captum
from captum.attr import Lime, IntegratedGradients
from captum.attr._core.lime import get_exp_kernel_similarity_function
from captum.attr._utils.visualization import visualize_image_attr
from captum._utils.models.linear_model import SkLearnLinearRegression


In [ ]:
# ---------- TO TRAIN, SET TRUE ------------ #
# TO NOT TRAIN, SET FALSE AND MAKE SURE THAT FULL DATA IS BEING USED, OTHERWISE SET TRAIN TO TRUE #
train = True

In [ ]:
EXPECTED_DIR = 'Data'

if os.path.exists('Data.zip'):
    print(f"Found sample data! Unzipping it...")
    !unzip Data.zip
else:
    print("Full dataset execution branch triggered...")
    if os.path.exists("Data"):
        shutil.rmtree("Data")

    file_id = '1lb44uTj0zN48kqFTzLscsxzF_u9sfrB_'
    direct_url = f'https://drive.google.com/uc?id={file_id}'

    gdown.download(direct_url, 'Data.zip', quiet=False)

    os.makedirs('Data', exist_ok=True)
    !unzip -o -q Data.zip -d Data

    print(f"Total images extracted: {len(os.listdir(EXPECTED_DIR))}")

###Creation of the Dataframe
After successfully importing the data, a DataFrame was created from the image filenames, resulting in 10,426 entries, where each row represents a single image. The DataFrame contains three columns: filename, which stores the image file name; generator_name, which identifies the source of the image (e.g., REAL, DALL-E, DEEPFACELAB); and binary_label, which encodes the target variable (1 for real images and 0 for fake images). If the generator_name is REAL we set the binary label as 1 else the binary label is set to 0. The binary_label column is the most important for this task, as it is used for training the classification model.


In [ ]:
# This block constructs a structured dataset from raw image files.
# Each filename contains information about whether the image is real or AI-generated.

output_dir = 'Data'

print(f"{len(os.listdir(output_dir))}")

filenames = os.listdir(output_dir)

dataset_rows = []

for f in filenames:
  if f.lower().endswith(('.jpg','.jpeg','.png')):
    parts = f.split('_')

    generator_name = parts[0].upper()

    binary_label = 1 if generator_name == 'REAL' else 0

    dataset_rows.append({
        'filename':f,
        'generator_name':generator_name,
        'binary_label':binary_label
    })

data = pd.DataFrame(dataset_rows)
print(data)

## 3. Data Exploration and Preprocessing

```markdown
## Contribution Metadata
Author(s): Emily Jenks
Section: Preprocessing part 3
Method: [label_distribution, undersampling, data_balancing, shuffling]
SHORT Description of contribution:
In this section, I analyzed the distribution of class labels in the dataset and identified a slight imbalance
between real and fake images, which was visualized using a pie chart.
To address the imbalance, I applied undersampling by randomly reducing the number of real images to match
the number of fake images, creating a balanced dataset.
Finally, I merged and shuffled the data to remove any ordering bias and
ensure randomness during model training.

Difficulty: BASIC
```
In this section, data exploration and preprocessing were conducted to better understand the dataset and prepare it for model training. The distribution of class labels was first analyzed, revealing a slight imbalance between real and AI-generated images. This imbalance was visualized using a pie chart to provide a clear overview of the dataset composition.

To address this issue, an undersampling technique was applied. Specifically, the number of real images was randomly reduced to match the number of fake images, resulting in a balanced dataset. This step is important to prevent the model from becoming biased toward the majority class and to ensure more reliable learning.

Finally, the data was merged and shuffled to eliminate any potential ordering bias and to introduce randomness during training. This helps improve the robustness of the model by ensuring that it does not learn spurious patterns related to data ordering.

Overall, these steps contribute to a more balanced and well-prepared dataset, which is essential for achieving fair and effective model performance.



### Label distribution
Initially the dataset was slighly imbalanced, with 5,890 real images and 4,536 fake images. In order to better understand the distribution, a pie chart was used to visualize the proportion of real and fake samples.



In [ ]:
# Separate the dataset into real and fake images to analyze class distribution.
# Find total number of Real and Fake images to determine whether the dataset is balanced or imbalanced.
real_images = data[data['binary_label']==1]
fake_images = data[data['binary_label']==0]

print(f"Total Real Images: {len(real_images)}")
print(f"Total Fake Images: {len(fake_images)}")

# Visualize the total number of Real and Fake images.
labels = ['Real Images', 'Fake Images']
sizes = [len(real_images), len(fake_images)]

plt.figure(figsize=(6, 6))
plt.pie(
    sizes,
    labels=labels,
    colors= ['pink', 'skyblue'],
    autopct='%1.1f%%',
    startangle=90
)

plt.title('Label Distribution')
plt.axis('equal')
plt.show()

### Undersampling
To address this class imbalance, undersampling was applied by randomly reducing the majority class (real images) to match the size of the minority class (fake images). This resulted in a balanced dataset of 4,536 images per class, ensuring that the model does not become biased toward the dominant class and the model's evaluation metrics are more reliable. The two classes were then merged and shuffled to remove any ordering bias and improve randomness during training.




In [ ]:
# Perform undersampling by randomly selecting a subset of real images equal to the number of fake images.
# A fixed random seed is used to ensure reproducibility.
real_images_under = real_images.sample(
    n=fake_images.shape[0],
    random_state= SEED
)

real_images = real_images_under

print("After undersampling:")
print("Real images:", real_images.shape[0])
print("Fake images:", fake_images.shape[0])


In [ ]:
# After balancing both classes, we merge them into a single dataset.
data_balanced = pd.concat([real_images, fake_images], axis=0)

# Shuffle the dataset randomly to remove any ordering bias.
data_balanced = data_balanced.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

print("Final balanced dataset:")
print(data_balanced['binary_label'].value_counts())


```markdown
## Contribution Metadata
Author(s): Chiara Bosi
Section: Preprocessing part 4
Method: [missing_value_check, image_size_check, dataset_splitting]
SHORT Description of contribution:
I contributed to the preprocessing pipeline by checking for missing values and duplicated entries, analyzing image resolutions and implementing the dataset splitting procedure.
Particular attention was given to dataset consistency and to maintaining a clear separation between training, validation and test sets in order to avoid data leakage.

Difficulty: BASIC
```

### Missing values Check

Before moving forward with the preprocessing pipeline, the dataset was inspected for missing values and duplicate entries. This step is necessary to ensure dataset integrity: incomplete samples could cause errors during training, while duplicate entries are particularly problematic if the same image ends up in both the training and evaluation splits, leading to an overly optimistic estimate of model performance.


In [ ]:
# Check for missing values in each column and count duplicate rows in the dataset.
print("\nMissing values per column:")
print(data_balanced.isnull().sum())

print("\nNumber of duplicated rows:")
print(data_balanced.duplicated().sum())

No missing values or duplicate entries were found, so no further cleaning was needed at this stage. The dataset is ready for image-level preprocessing and model training.


### Image Size Check

The next step was to check whether the images in the dataset share a consistent spatial resolution.
Since CNNs require fixed-size inputs, all images must be resized to a common dimension before training. Additionally, inspecting the original image sizes  helps identify potentially corrupted or unusually small images that could negatively affect the learning process.

In [ ]:
# Verify that all image files are readable and not corrupted.
# For readable files, record width, height and color mode
# Log corrupted files separately
IMAGE_DIR = "Data"

image_info = []
bad_files = []

for _, row in data_balanced.iterrows():
    img_path = os.path.join(IMAGE_DIR, row["filename"])

    try:
        with Image.open(img_path) as img:
            width, height = img.size
            mode = img.mode

        image_info.append({
            "filename": row["filename"],
            "width": width,
            "height": height,
            "mode": mode,
            "binary_label": row["binary_label"],
            "generator_name": row["generator_name"]
        })

    except (UnidentifiedImageError, OSError) as e:
        bad_files.append((row["filename"], str(e)))

image_sizes = pd.DataFrame(image_info)

print("Number of readable images:", len(image_sizes))
print("Number of problematic files:", len(bad_files))

if bad_files:
    print("\nProblematic files:")
    for f, err in bad_files[:10]:
        print(f, "->", err)

# Verify that all images are of the same size
image_sizes["size"] = (
    image_sizes["width"].astype(str) + "x" + image_sizes["height"].astype(str)
)

size_counts = (
    image_sizes["size"]
    .value_counts()
    .reset_index()
)

size_counts.columns = ["Image size", "Count"]

display(size_counts.style.hide(axis="index"))




All images in the dataset share the same spatial resolution (256×256 pixels) and no problematic files were detected.

Since the dataset is already uniform in terms of image dimensions, no additional resizing is required before training the models.

### Dataset Split into Training, Testing, and Validation

The dataset was divided into training, validation, and test subsets using a 70% / 15% / 15% split strategy.

- The **training set**  (70%) was used for model optimization;
- The **validation set** (15%) was used for hyperparameter tuning and model selection;
- The **test set** (15%) was held out and used exclusively for the final evaluation and explainability analysis.

Maintaining a strict separation between these subsets is important to avoid data leakage and ensure that the results reflect the true generalization capability of the models on unseen images.

The 70/15/15 ratio was chosen as a reasonable trade-off: the larger training portion gives the models enough data to learn from, while two equal evaluation splits allow for both reliable tuning and honest final assessment. Stratification is applied at each split to ensure that the class balance is preserved across all three subsets.

To guarantee reproducibility and prevent data leakage, the three splits are saved alongside the model weights. This way, when the notebook is run with `train = False`, the exact same training, validation and test partitions are reloaded, ensuring that the saved model weights are always tested on the same data.


In [ ]:
print(data_balanced)

In [ ]:
# Step 1: split 70% train, 30% temporary test data
if train == True:
  train_data, temp_data = train_test_split(
    data_balanced,
    test_size=0.30,
    stratify=data_balanced['binary_label'],
    random_state=SEED
  )

# Step 2: split the remaining data equally into 15% validation and 15% test sets
  val_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    stratify=temp_data['binary_label'],
    random_state=42
  )

# Reset indexes so all three DataFrames are consistently indexed from 0.
  train_data = train_data.reset_index(drop=True)
  val_data = val_data.reset_index(drop=True)
  test_data = test_data.reset_index(drop=True)

  os.makedirs('/content/fixed_data', exist_ok=True)
  train_data.to_csv(f"/content/fixed_data/train_data.csv", index=False)
  val_data.to_csv(f"/content/fixed_data/val_data.csv", index=False)
  test_data.to_csv(f"/content/fixed_data/test_data.csv", index=False)

  !zip -r /content/fixed_data.zip {'/content/fixed_data'}

  print("Downloading zip file to your local computer...")
  files.download('/content/fixed_data.zip')

else:
  # Load the pre-saved splits to ensure all models are evaluated on exactly the same data partitions

  DATA_ZIP_FILE_ID = "1v9dtwoOswQcCGI-LM1pLDAKvCPNzNElB"
  DATA_ZIP_DRIVE_SHARE_LINK = f"https://drive.google.com/uc?id={DATA_ZIP_FILE_ID}"
  DATA_LOCAL_ZIP_PATH = "/content/fixed_data.zip"
  WEIGHTS_LOCATION = "/content/fixed_data"

  gdown.download(DATA_ZIP_DRIVE_SHARE_LINK, DATA_LOCAL_ZIP_PATH, quiet=False)
  !unzip -o {DATA_LOCAL_ZIP_PATH} -d /

  train_data = pd.read_csv('/content/fixed_data/train_data.csv')
  val_data = pd.read_csv('/content/fixed_data/val_data.csv')
  test_data = pd.read_csv('/content/fixed_data/test_data.csv')

In [ ]:
print(train_data)
print(val_data)
print(test_data)

In [ ]:
print("Train size:", train_data.shape[0])
print("Validation size:", val_data.shape[0])
print("Test size:", test_data.shape[0])

print("\nTrain label distribution:")
print(train_data['binary_label'].value_counts())

print("\nValidation label distribution:")
print(val_data['binary_label'].value_counts())

print("\nTest label distribution:")
print(test_data['binary_label'].value_counts())

## 3. Baseline Model - SHALLOW CNNs

```markdown
## Contribution Metadata
Author(s): Chiara Bosi
Section: Shallow CNNs part 1
Method: [reproducibility, data_augmentation, helper_functions]
SHORT Description of contribution:
I contributed to the implementation of the experimental pipeline by configuring reproducibility settings, defining the data augmentation strategy and developing reusable helper functions for training and evaluation.
These components were designed to improve experiment consistency, reduce code duplication and make the overall training pipeline more modular and reproducible.

Difficulty: INTERMEDIATE
```


To ensure reproducibility across runs, fixed random seeds were defined for NumPy, PyTorch, and Python’s `random` module.

Without this, operations such as weight initialization, batch shuffling, and random augmentation would produce different results each time the notebook is executed, making it harder to compare experiments reliably.

The execution device was also configured to use GPU acceleration whenever CUDA is available, while maintaining compatibility with CPU execution environments.

In [ ]:
# Set random seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

#Use GPU if it is available, otherwise CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

#Global Variables for Image Directory and Image Size
IMAGE_DIR = "Data"
IMG_SIZE = (224, 224)

### Data Augmentation

Data augmentation was applied during training to improve generalization and reduce overfitting. Given that the training set is relatively small, augmenting the data allows the models to see more varied versions of each image, making them less likely to memorize specific patterns.

The augmentation pipeline includes random horizontal flipping, small rotations (up to 15°), and slight color perturbations affecting brightness, contrast, and saturation. These transformations preserve the semantic content of the images while introducing controlled variability in orientation, illumination and appearance.

Note that augmentation techniques were applied only to the training set, while validation and test transforms include only resizing and normalization, ensuring that evaluation is always performed on unmodified images.

In [ ]:
# Created a class which will load, decode and map images to the tensor.
class ImageDataset(Dataset):
    """Custom Dataset for loading images from the dataframe."""
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.df.iloc[idx]["filename"])
        image = Image.open(img_path).convert("RGB")
        label = torch.tensor(self.df.iloc[idx]["binary_label"], dtype=torch.float32) #Binary Target Label mapping

        if self.transform:
            image = self.transform(image)

        return image, label

# Two separate transform pipelines:
# - train_transform: includes augmentation to improve generalization
# - eval_transform: resize + normalize only, used for validation and test

train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], #ImageNet channel means
                         [0.229, 0.224, 0.225]) #ImageNet Channel Standard Deviation
])

eval_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


### Helper Functions

To keep the training pipeline clean and avoid repeating the same code across different experiments, a set of shared helper functions was defined.

These functions handle the main operations needed throughout the project:
- training the model for one epoch;
- evaluating a model on a given split;
- computing classification metrics;
- configuring optimization strategies;
- visualizing confusion matrices.

In [ ]:
#Train a model for one epoch, updating the model weights after each batch
def train_one_epoch(model, loader, optimizer, criterion):

    model.train()
    total_loss = 0.0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

#Evaluate the model using loss, accuracy, AUC and F1-score.
def evaluate(model, loader, criterion):

    model.eval()
    all_probs, all_labels = [], []
    total_loss = 0.0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(outputs).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy())

    preds = (np.array(all_probs) >= 0.5).astype(int)

    return {
        "loss": total_loss / len(loader),
        "accuracy": accuracy_score(all_labels, preds),
        "auc": roc_auc_score(all_labels, all_probs),
        "f1": f1_score(all_labels, preds)
    }

# Build an Adam or SGD optimizer.
def get_optimizer(model, optimizer_name, lr, only_trainable=False):

    params = model.parameters()
    if only_trainable:
        params = filter(lambda p: p.requires_grad, model.parameters())

    if optimizer_name == "adam":
        return torch.optim.Adam(params, lr=lr)
    elif optimizer_name == "sgd":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

# Plot a confusion matrix for a trained classifier.
def plot_confusion_matrix(model, loader, title, cmap="Blues"):

    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(DEVICE)).squeeze(1)
            preds = (torch.sigmoid(outputs) >= 0.5).long().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    cm = confusion_matrix(all_labels, all_preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=["AI", "Real"])
    disp.plot(cmap=cmap)
    plt.title(title)
    plt.show()


```markdown
## Contribution Metadata
Author(s): Ludovica Tapparo
Section: Shallow CNNs part 2
Method: [shallow_cnns, conv_layers, sequential_layers]
SHORT Description of contribution: My contribution to this section was to develop the
code for the shallow CNN baseline architectures and to describe how CNNs process image
data, interpret the structure of `CNN-Small` and `CNN-Medium`, and the purpose of
their main components. I also motivated the use of two CNN variants as a controlled
baseline comparison for studying the effect of increasing model depth
before introducing ResNet18.


Difficulty: INTERMEDIATE
```
### CNN Definition and motivation

A Convolutional Neural Network (CNN) is a neural network architecture specifically designed for image data. Unlike fully connected networks, which treat an image as a flat vector of pixels, CNNs preserve the spatial structure of the image and learn visual patterns through convolutional filters. These filters slide across the image and detect local features such as edges, textures, shapes, and more complex visual structures. As the image passes through deeper convolutional layers, the network can progressively learn more abstract representations: earlier layers usually capture low-level details, while later layers combine them into higher-level features useful for classification.

CNNs are appropriate for this project because the task is based on distinguishing real images from AI-generated or deepfake images. This type of classification depends on visual information, including texture consistency, local artifacts, edges, and subtle image patterns that may not be easily visible to humans. By using convolutional layers, the model can automatically learn these discriminative visual features directly from the images instead of relying on manually defined features.

The reason why we chose to try shallow CNNs as our baseline architectures lies within the fact that they are the standard model choice that is associated to image data. Therefore, they provide a simple, interpretable, and task-appropriate reference point for image classification. Since CNNs are specifically designed to process images, even a shallow CNN is capable of learning meaningful visual features. At the same time, these models are much less complex than deeper pretrained architectures such as ResNet18, which makes them suitable for establishing an initial performance benchmark.

In this section, we define our shallow CNN models from scratch in order to establish the baseline architectures before comparing them with a deeper ResNet18 model.




In [ ]:
# We define a simple CNN architecture from scratch.
# The same class is used to build two variants:
# - CNN-Small: 2 convolutional blocks
# - CNN-Medium: 3 convolutional blocks
class ShallowCNN(nn.Module):

    def __init__(self, num_blocks=2, dropout=0.5):
        super().__init__()

        if num_blocks not in [2, 3]:
            raise ValueError("num_blocks must be either 2 or 3 for this experiment.") # We test only the two specified variants

        channels = [3, 32, 64, 128] # Number of channels used by each conv. block. Input images have 3 because they are RGB
        layers = []

        # Each block applies: convolution + batch normalization + ReLU + max pooling
        for block_idx in range(num_blocks):
            layers.extend([
                nn.Conv2d(channels[block_idx], channels[block_idx + 1], kernel_size=3, padding=1),
                nn.BatchNorm2d(channels[block_idx + 1]),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=2, stride=2)
            ])

        self.features = nn.Sequential(*layers) # Feature extractor part of the network
        self.pool = nn.AdaptiveAvgPool2d((1, 1)) # Reduces each feature map to a single value

        # Final classification layer
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=dropout),
            nn.Linear(channels[num_blocks], 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

# Function that builds the two CNN variants
def build_shallow_cnn(num_blocks=2, dropout=0.5):
    return ShallowCNN(num_blocks=num_blocks, dropout=dropout)


In [ ]:
# Shallow CNN with 2 blocks
CNN_block2 = build_shallow_cnn(num_blocks=2, dropout=0.5)
summary(CNN_block2)

The summary above shows the structure of `CNN-Small`, which is composed of two convolutional blocks followed by an adaptive average pooling layer and a final classifier. The first convolutional layer takes the 3 RGB input channels and produces 32 feature maps, while the second increases the representation from 32 to 64 feature maps. Each convolution is followed by batch normalization, ReLU activation, and max pooling, which together help stabilize training, introduce non-linearity, and reduce the spatial size of the feature maps. Layers such as ReLU, MaxPool, AdaptiveAvgPool, Flatten, and Dropout do not have trainable parameters, while convolutional, batch normalization, and linear layers do. The final linear layer outputs a single logit for binary classification. Overall, `CNN-Small` has 19,649 trainable parameters and 0 non-trainable parameters, confirming that it is a lightweight baseline model whose weights are fully updated during training.


In [ ]:
# Shallow CNN with 3 blocks
CNN_block3 = build_shallow_cnn(num_blocks=3, dropout=0.5)
summary(CNN_block3)

The `CNN-Medium` summary follows the same general structure as `CNN-Small`, but includes an additional third convolutional block. The added third block further increases the number of channels from 64 to 128, giving the model a higher representational capacity and allowing it to learn more complex visual patterns. This additional `Conv2d` layer introduces 73,856 trainable parameters, followed by a `BatchNorm2d` layer with 256 parameters. Because the final feature vector now has 128 channels instead of 64, the final linear layer has 129 parameters. Overall, `CNN-Medium` has 93,825 trainable parameters, compared to 19,649 in `CNN-Small`, showing that the extra block substantially increases the model capacity while still keeping the architecture relatively simple.


```markdown
## Contribution Metadata
Author(s): Emily Jenks
Section: Shallow CNNs part 3
Method: [cnn_training, validation_pipeline, model_comparison, metric_evaluation, reproducibility_loading]
SHORT Description of contribution:
In this section, I implemented a full training and validation pipeline for two shallow CNN models with different
numbers of convolutional blocks (2 vs 3).
Both models were trained using identical hyperparameters to ensure a fair comparison, with the only difference being model depth.
After training, I evaluated both models using validation metrics such as loss, accuracy, AUC, and F1-score,
and directly compared their performance to measure the impact of adding an extra convolutional block.
I also stored the results as CSV files for later analysis and included functionality to reload saved results
and model outputs when training is not performed.

Difficulty: INTERMEDIATE
```


##Shallow CNN Baseline Training, Evaluation, and Comparison Pipeline

This section presents the complete pipeline for training, validating, and comparing two shallow CNN baseline models with different depths (2 vs 3 convolutional blocks). It includes the full workflow from model training and evaluation to metric comparison, result storage, and reproducibility through saved model artifacts. The goal is to isolate the effect of increasing network depth while keeping all other hyperparameters fixed for a fair performance comparison.

In [ ]:
# If not training, download the CNN results and the model weights, and unzip the file
if train == False:
  ZIP_FILE_ID = "1Sxdt-XZM9tmB_yFaE9cqEJszNfuUSX_s"
  ZIP_DRIVE_SHARE_LINK = f"https://drive.google.com/uc?id={ZIP_FILE_ID}"
  LOCAL_ZIP_PATH = "/content/shallow_cnn_baselines.zip"
  WEIGHTS_LOCATION = "/content/shallow_cnn_weights"

  gdown.download(ZIP_DRIVE_SHARE_LINK, LOCAL_ZIP_PATH, quiet=False)
  !unzip -o {LOCAL_ZIP_PATH} -d /

  # Load saved CNN results from CSV files into dataframes
  cnn_small_results = pd.read_csv(f"{WEIGHTS_LOCATION}/cnn_small_results.csv")
  cnn_medium_results = pd.read_csv(f"{WEIGHTS_LOCATION}/cnn_medium_results.csv")

  # Convert results into dictionary format for easier access later in the notebook
  cnn_small_result = cnn_small_results.iloc[0].to_dict()
  cnn_medium_result = cnn_medium_results.iloc[0].to_dict()

This code above handles the inference-time setup when training is disabled by loading previously saved CNN results and model artifacts. When `train` is set to False, it downloads a zipped file containing stored outputs from Google Drive using a file ID and a generated download URL. The file is saved locally as a ZIP archive in the Colab environment. It is then extracted using the `unzip` command into the root directory, restoring the saved project files, including model results and weights. After extraction, the code reads the saved CNN evaluation results from CSV files located in the `shallow_cnn_weights` directory. These CSV files are loaded into Pandas DataFrames for both the small (2-block) and medium (3-block) CNN models. The DataFrames are then converted into dictionaries using `.iloc[0].to_dict()`, restoring the original result structure used in the training pipeline. This allows the rest of the codebase to treat loaded results exactly like freshly computed outputs. Overall, this block enables reproducibility by allowing previously trained models and their evaluation metrics to be reloaded without retraining.

In [ ]:
# EXPERIMENTAL HYPERPARAMETERS
# Fixed training settings for the shallow CNN baselines.
# These are not selected by random search.
CNN_FIXED_CONFIG = {
    "optimizer": "adam",
    "lr": 1e-3, #learning rate
    "batch_size": 32,
    "dropout": 0.5,
    "validation_epochs": 10,
    "final_epochs": 15,
}


In this code below, this function defines a full training and validation pipeline for a shallow CNN with a fixed hyperparameter configuration defined above. It begins by copying a predefined configuration dictionary to ensure consistent training settings. The function then creates training and validation datasets using an `ImageDataset` class, applying appropriate transformations for training and evaluation. These datasets are wrapped into PyTorch `DataLoader` objects, which handle batching, shuffling, and efficient data loading during training. Next, the CNN model is constructed using `build_shallow_cnn`, where the only architectural variable is the number of convolutional blocks, and the model is moved to the available device (CPU or GPU). A binary classification loss function, `BCEWithLogitsLoss`, is used to compute training error, and an optimizer is initialized using a helper function based on the selected optimizer type and learning rate. The model is then trained for a fixed number of epochs using a loop that calls `train_one_epoch`, with loss printed after each epoch for monitoring. After training, the model is evaluated on the validation set using an `evaluate` function that computes metrics such as loss, accuracy, AUC, and F1-score. These metrics are stored in a structured dictionary along with the model name, architecture depth, and hyperparameters. The function also prints a summary of validation performance for quick inspection. Finally, it returns both the evaluation results and the trained model, enabling later comparison or saving.

In [ ]:
# Train one shallow CNN with fixed hyperparameters and evaluate it on validation data.
# This function builds the dataset, trains the model, and evaluates performance on validation data
def train_and_validate_shallow_cnn(model_name, num_blocks, train_data, val_data, image_dir):

    # Copy fixed hyperparameters
    config = CNN_FIXED_CONFIG.copy()

    # Create training and validation datasets from image data
    train_ds = ImageDataset(train_data, image_dir, train_transform)
    val_ds = ImageDataset(val_data, image_dir, eval_transform)

    # Data Routing Loaders
    # Set up data loaders to feed batches of data into the model during training and validation
    train_loader = DataLoader(
        train_ds,
        batch_size=config["batch_size"],
        shuffle=True
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=config["batch_size"],
        shuffle=False
    )

    # Model Initialization
    # Initialize the shallow CNN model using the specified number of convolutional blocks
    # Move the model to the correct device (GPU or CPU)
    model = build_shallow_cnn(
        num_blocks=num_blocks,
        dropout=config["dropout"]
    ).to(DEVICE)

    # Define loss function and optimizer for training
    criterion = nn.BCEWithLogitsLoss()
    optimizer = get_optimizer(
        model,
        config["optimizer"],
        config["lr"]
    )

    # Train the model for a fixed number of epochs using forward and backward propagation
    print(f"\n===== Training fixed baseline: {model_name} =====")
    for epoch in range(config["validation_epochs"]):
        loss = train_one_epoch(model, train_loader, optimizer, criterion)
        print(f"Epoch {epoch + 1}/{config['validation_epochs']} — loss: {loss:.4f}")

    # Performance logging
    # Evaluate model performance on the validation set and compute key metrics
    val_metrics = evaluate(model, val_loader, criterion) #dictionary containing loss, accuracy, AUC, f1 score

    # Store relevant results and hyperparameters in a dictionary for later comparison
    result = {
        "model": model_name,
        "num_blocks": num_blocks,
        "optimizer": config["optimizer"],
        "lr": config["lr"],
        "batch_size": config["batch_size"],
        "dropout": config["dropout"],
        "val_auc": round(val_metrics["auc"], 4),
        "val_acc": round(val_metrics["accuracy"], 4),
        "val_f1": round(val_metrics["f1"], 4),
    }

    # Print validation performance summary
    print(
        f"Validation — AUC: {val_metrics['auc']:.4f} | "
        f"Acc: {val_metrics['accuracy']:.4f} | "
        f"F1: {val_metrics['f1']:.4f}"
    )

    return result, model


This code trains and validates two shallow CNN baseline models when the `train` flag is set to True. It calls the function `train_and_validate_shallow_cnn` twice, once for a 2-block CNN and once for a 3-block CNN. Both models use the same training setup, including identical hyperparameters and datasets, ensuring a fair comparison where the only controlled variable is the number of convolutional blocks. The first call trains the “CNN-Small-2ConvBlocks” model using two convolutional layers and returns both its evaluation results and trained validation model. The second call trains the “CNN-Medium-3ConvBlocks” model with three convolutional blocks and similarly returns its results and model. Both models are trained using the same training and validation datasets, as well as the same image directory. This shows the effect of increasing model depth on performance. The outputs include performance metrics and the trained model objects, which can later be used for evaluation, comparison, or saving.

In [ ]:
# Train and validate the two shallow CNN baselines without random search.
# Both models use exactly the same hyperparameters; only the number of convolutional blocks changes.
if train == True:
  cnn_small_result, cnn_small_val_model = train_and_validate_shallow_cnn(
    model_name="CNN-Small-2ConvBlocks",
    num_blocks=2,
    train_data=train_data,
    val_data=val_data,
    image_dir=IMAGE_DIR
  )

  cnn_medium_result, cnn_medium_val_model = train_and_validate_shallow_cnn(
    model_name="CNN-Medium-3ConvBlocks",
    num_blocks=3,
    train_data=train_data,
    val_data=val_data,
    image_dir=IMAGE_DIR
  )



This code saves the results of two trained shallow CNN models when training is enabled. It first checks if the `train` flag is set to True, then creates a local folder called `shallow_cnn_weights` using `os.makedirs`, ensuring the folder exists and if it already exists,  no error will be raised. The results for the small and medium CNN models, originally stored as dictionaries, are converted into Pandas DataFrames. Each DataFrame contains the model’s evaluation metrics and configuration details. These DataFrames are then exported as CSV files into the created directory. The `index=False` parameter is used to prevent Pandas from writing row indices into the files, ensuring clean output format. This process enables persistent storage of model evaluation results for later analysis or comparison.

In [ ]:
# If training, save the results of two trained Shallow CNNs in a folder
if train == True:
  LOCAL_EXPORT_DIR = "/content/shallow_cnn_weights"
  os.makedirs(LOCAL_EXPORT_DIR, exist_ok=True)

  # Convert stored result dictionaries into dataframes for saving
  df_small_result = pd.DataFrame([cnn_small_result])
  df_medium_result = pd.DataFrame([cnn_medium_result])

  # Save the results as CSV files for later use or comparison
  df_small_result.to_csv(f"{LOCAL_EXPORT_DIR}/cnn_small_results.csv", index=False)
  df_medium_result.to_csv(f"{LOCAL_EXPORT_DIR}/cnn_medium_results.csv", index=False)

The code below compares two shallow CNN models: one with 2 convolutional blocks and one with 3 convolutional blocks. It first combines their validation results into a single DataFrame and adds a readable label for each architecture. It then displays key performance metrics such as accuracy, AUC, and F1-score alongside important hyperparameters. Next, it extracts the results for each model and computes the differences in performance metrics between the 3-block and 2-block CNNs. These differences are stored in a new DataFrame to clearly show whether adding an extra convolutional block improves performance. Finally, the 3-block CNN is selected as the preferred baseline model for further comparison with more advanced architectures like ResNet18.

In [ ]:
# METRIC COMPARISON & VARIANCE ANALYSIS
# Keep the order 2 blocks -> 3 blocks so the improvement is easy to read.
# Consolidate validation results for both architectures
cnn_validation_results = pd.DataFrame([cnn_small_result, cnn_medium_result])
cnn_validation_results["architecture"] = ["2 convolutional blocks", "3 convolutional blocks"]

print("===== Fixed shallow CNN validation results: 2 blocks vs 3 blocks =====")
display(cnn_validation_results[["model", "architecture", "val_acc", "val_auc", "val_f1", "optimizer", "lr", "batch_size", "dropout"]])

# Extract rows for direct comparison between the two CNN configurations
val_small = cnn_validation_results[cnn_validation_results["num_blocks"] == 2].iloc[0]
val_medium = cnn_validation_results[cnn_validation_results["num_blocks"] == 3].iloc[0]

# Check the validation improvement from 2 convolution block to 3 convolution blocks shalllow CNN
cnn_validation_improvement = pd.DataFrame([{
    "comparison": "CNN-Medium-3ConvBlocks minus CNN-Small-2ConvBlocks",
    "accuracy_change": round(val_medium["val_acc"] - val_small["val_acc"], 4),
    "auc_change": round(val_medium["val_auc"] - val_small["val_auc"], 4),
    "f1_change": round(val_medium["val_f1"] - val_small["val_f1"], 4),
}])

print("===== Validation improvement from 2 blocks to 3 blocks =====")
display(cnn_validation_improvement)

# Saving the best Shallow CNN model if needed in the future
best_config_cnn = cnn_medium_result.copy()
print("Main shallow CNN baseline for comparison with ResNet18: CNN-Medium-3ConvBlocks")


```markdown
## Contribution Metadata
Author(s): Vatsalya Arora
Section: Shallow CNNs part 4
Method: [train_and_test, test_results, confusion_matrix]
SHORT Description of contribution: This section handles the final testing phase for our two shallow CNN baselines.
To ensure a fair and comprehensive test, we combine our training and validation datasets into a single pool to retrain the models from scratch
(if train == True), save their weights, and download them as a zip file.

If we skip training, the code automatically bypasses that loop, loads the pre-saved weights (.pth files)
into the respective model shells, and initializes the test loaders.
Finally, it evaluates both architectures on the unseen test dataset, calculates the exact performance improvements g
ained by adding the third convolutional block, and displays the comparative results using clean metrics tables and confusion matrices.

Difficulty: INTERMEDIATE
```
### Final training of both Shallow CNNs

In [ ]:
# Final training of both shallow CNN baselines using train + validation data.
# This gives a fair held-out test comparison for CNN-Small and CNN-Medium.
full_train = pd.concat([train_data, val_data]).reset_index(drop=True)
criterion = nn.BCEWithLogitsLoss()

full_ds_cnn = ImageDataset(full_train, IMAGE_DIR, train_transform)
test_ds_cnn = ImageDataset(test_data, IMAGE_DIR, eval_transform)

def train_final_shallow_cnn(config):
    full_loader = DataLoader(
        full_ds_cnn,
        batch_size=int(config["batch_size"]),
        shuffle=True
    )
    test_loader = DataLoader(
        test_ds_cnn,
        batch_size=int(config["batch_size"]),
        shuffle=False
    )

    model = build_shallow_cnn(
        num_blocks=int(config["num_blocks"]),
        dropout=float(config["dropout"])
    ).to(DEVICE)

    optimizer = get_optimizer(
        model,
        config["optimizer"],
        float(config["lr"])
    )

    print(f"===== Final training: {config['model']} =====")
    for epoch in range(CNN_FIXED_CONFIG["final_epochs"]):
        loss = train_one_epoch(model, full_loader, optimizer, criterion)
        print(f"Epoch {epoch + 1}/{CNN_FIXED_CONFIG['final_epochs']} — loss: {loss:.4f}")

    return model, test_loader

In [ ]:
if train == True:
  final_cnn_configs = [cnn_small_result, cnn_medium_result]
  final_cnn_models = {}
  final_cnn_loaders = {}
  final_cnn_metrics = {}
  final_cnn_rows = []

  for config in final_cnn_configs:
      model, loader = train_final_shallow_cnn(config)

      final_cnn_models[config["model"]] = model
      final_cnn_loaders[config["model"]] = loader

      torch.save(model.state_dict(), f"{LOCAL_EXPORT_DIR}/{config['model']}.pth")
  print("Weights saved locally inside Colab environment.")

In [ ]:
if train == True:
  !zip -r /content/shallow_cnn_baselines.zip {LOCAL_EXPORT_DIR}

  print("Downloading zip file to your local computer...")
  files.download('/content/shallow_cnn_baselines.zip')

### Shallow CNNs Testing and final evaluation


In [ ]:
final_cnn_configs = [cnn_small_result, cnn_medium_result]
final_cnn_models = {}
final_cnn_metrics = {}
final_cnn_rows = []
final_cnn_loaders = {}

for config in final_cnn_configs:
  model = build_shallow_cnn(
    num_blocks=int(config["num_blocks"]),
    dropout=float(config["dropout"])
  ).to(DEVICE)
  saved_state_dict = torch.load(f'/content/shallow_cnn_weights/{config["model"]}.pth', map_location=DEVICE)
  model.load_state_dict(saved_state_dict)
  test_loader = DataLoader(
        test_ds_cnn,
        batch_size=int(config["batch_size"]),
        shuffle=False
    )
  final_cnn_loaders[config["model"]] = test_loader

  test_metrics = evaluate(model, test_loader, criterion)
  final_cnn_metrics[config["model"]] = test_metrics
  final_cnn_rows.append({
        "model": config["model"],
        "num_blocks": config["num_blocks"],
        "architecture": f"{int(config['num_blocks'])} convolutional blocks",
        "test_accuracy": test_metrics["accuracy"],
        "test_auc": test_metrics["auc"],
        "test_f1": test_metrics["f1"]
    })
  final_cnn_models[config["model"]] = model
print(f"{final_cnn_rows}")



In [ ]:
cnn_test_results = pd.DataFrame(final_cnn_rows)
cnn_test_results[["test_accuracy", "test_auc", "test_f1"]] = cnn_test_results[["test_accuracy", "test_auc", "test_f1"]].round(4)

print("===== Shallow CNN test results: 2 blocks vs 3 blocks =====")
display(cnn_test_results[["model", "architecture", "test_accuracy", "test_auc", "test_f1"]])

# Explicitly quantify the test improvement obtained by adding the third convolutional block.
test_small = cnn_test_results[cnn_test_results["num_blocks"] == 2].iloc[0]
test_medium = cnn_test_results[cnn_test_results["num_blocks"] == 3].iloc[0]

cnn_test_improvement = pd.DataFrame([{
    "comparison": "CNN-Medium-3ConvBlocks minus CNN-Small-2ConvBlocks",
    "accuracy_change": round(test_medium["test_accuracy"] - test_small["test_accuracy"], 4),
    "auc_change": round(test_medium["test_auc"] - test_small["test_auc"], 4),
    "f1_change": round(test_medium["test_f1"] - test_small["test_f1"], 4),
}])

print("===== Test improvement from 2 blocks to 3 blocks =====")
display(cnn_test_improvement)

# Store the 3-block CNN as the main shallow CNN baseline for direct comparison with ResNet18.
best_model_cnn = final_cnn_models["CNN-Medium-3ConvBlocks"]
test_loader_cnn = final_cnn_loaders["CNN-Medium-3ConvBlocks"]
test_metrics_cnn = final_cnn_metrics["CNN-Medium-3ConvBlocks"]

In [ ]:
# Detailed test evaluation for both shallow CNN baselines.
# This makes the depth comparison explicit: 2 convolutional blocks vs 3 convolutional blocks.
print(f"=== BASELINE TEST RESULTS: SHALLOW CNNS BUILT FROM SCRATCH ===")
for _, row in cnn_test_results.iterrows():
    print(f"{row['model']} ({int(row['num_blocks'])} convolutional blocks)")
    print(f"Accuracy : {row['test_accuracy']:.4f}")
    print(f"AUC-ROC  : {row['test_auc']:.4f}")
    print(f"F1-score : {row['test_f1']:.4f}")

print("=== EFFECT OF ADDING THE THIRD CONVOLUTIONAL BLOCK ===")
print(f"Accuracy change: {cnn_test_improvement.loc[0, 'accuracy_change']:+.4f}")
print(f"AUC change     : {cnn_test_improvement.loc[0, 'auc_change']:+.4f}")
print(f"F1 change      : {cnn_test_improvement.loc[0, 'f1_change']:+.4f}")

if cnn_test_improvement.loc[0, "auc_change"] > 0:
    print("Result: the 3-block CNN improves over the 2-block CNN on AUC.")
else:
    print("Result: in this run, the 3-block CNN did not improve AUC. Re-run with the same fixed setup or discuss this as experimental variance.")

# Plot confusion matrices for both shallow CNNs so the comparison is visible.
plot_confusion_matrix(
    final_cnn_models["CNN-Small-2ConvBlocks"],
    final_cnn_loaders["CNN-Small-2ConvBlocks"],
    title="CNN-Small Baseline — 2 Conv Blocks",
    cmap="Oranges"
)

plot_confusion_matrix(
    final_cnn_models["CNN-Medium-3ConvBlocks"],
    final_cnn_loaders["CNN-Medium-3ConvBlocks"],
    title="CNN-Medium Baseline — 3 Conv Blocks",
    cmap="Oranges"
)

print("Baseline shallow CNN models saved.")


This output demonstrates a clear performance leap when expanding our baseline architecture from two convolutional blocks to three. The CNN-Medium model achieves an accuracy of 88.46% (a +9.11% improvement) and a stellar AUC-ROC of 0.9631, confirming that the added depth significantly enhances the network's capacity to extract robust structural features from the images.

Looking at the confusion matrices, the 3-block model drastically cuts down on misclassifications. It reduces False Positives (AI images misidentified as Real) from 58 to 38, and crucially slashes False Negatives (Real images misidentified as AI) from 223 down to 119. This massive reduction in errors boosts the F1-score by +11.23%, proving that the deeper baseline is far more balanced and reliable before we even introduce ResNet18.

## 4. Proposed Model - ResNet18

```markdown
## Contribution Metadata
Author(s): Vatsalya Arora
Section: ResNet18 part 1
Method: [transfer_learning, freeze_strategy]
SHORT Description of contribution:
I contributed to the project by developing the ResNet18 architecture and its transfer learning pipeline.
This work included justifying the shift from shallow CNNs to deeper pretrained networks, optimizing architectural choices, and configuring fine-tuning parameters.
The core analysis focused on the strengths of residual connections for deepfake detection,
the benefits of transfer learning on limited datasets, and the deployment of ResNet18 as a balanced solution between computational efficiency and classification accuracy.

Difficulty: INTERMEDIATE
```



After evaluating the shallow CNN architectures, we explored deeper convolutional neural networks, which are generally more effective at learning visual representations from complex image data.

Instead of training a deep neural network entirely from scratch, we opted for a transfer learning approach using pretrained models. Transfer learning is particularly effective when the available dataset is relatively limited, because pretrained convolutional filters already encode generic visual patterns such as edges, textures, shapes, and object-level structures learned from large-scale image datasets.
These visual features remain useful even for tasks different from ImageNet classification, as fine- tuning the network allows to adapt them to the considered task while improving training stability and convergence speed. In the context of real-versus-AI image classification, this is particularly useful because the distinction between real and synthetic images often depends on subtle visual patterns such as texture inconsistencies, facial irregularities, or localized synthetic artifacts.


In particular, we focused on the family of residual neural networks (**ResNets**), which are specifically designed to address the degradation problem that affects very deep convolutional models. ResNet architectures introduce shortcut residual connections that facilitate gradient propagation during training and help mitigate the commonly encountered vanishing gradient problem.

Among the different ResNet variants, ResNet18 was selected as the proposed architecture for this project.
Larger models such as ResNet50 were considered, but they introduced significantly higher computational cost and training time. ResNet18, instead, provided a better trade-off between efficiency and predictive performance, allowing faster computation while still achieving strong classification results.


After model selection, different fine-tuning strategies were implemented by selectively freezing portions of the pretrained network:
- **fc_only**: only the final classification head is trainable;
- **layer4**: the last residual block and the classification head are trainable;
- **none**: the entire network is trainable.

This setup allowed controlling the balance between preserving pretrained knowledge and adapting the model to the target dataset.

In [ ]:
def build_resnet18(dropout=0.5, freeze_strategy="fc_only"):

    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1) #loading the pre-trained ResNet model weights

    # Freeze all layers first
    for param in model.parameters():
        param.requires_grad = False

    if freeze_strategy == "layer4":
        for param in model.layer4.parameters():
            param.requires_grad = True
    elif freeze_strategy == "none":
        for param in model.parameters():
            param.requires_grad = True
    elif freeze_strategy != "fc_only":
        raise ValueError("freeze_strategy must be 'fc_only', 'layer4', or 'none'.")

    # Replace ImageNet classification head with binary classification head
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, 1)
    )

    return model


The `build_resnet18()` function initializes a pretrained ResNet18 architecture and adapts it to the binary classification task.

The pretrained ImageNet weights are loaded first, after which all layers are initially frozen by disabling gradient updates. The selected freezing strategy then determines which portions of the network remain trainable during fine-tuning.

Finally, the original ImageNet classification head is replaced with a custom binary classification head composed of:
- a dropout layer for regularization;
- a fully connected layer with a single output neuron.

The single output logit is later converted into class probabilities during evaluation through the sigmoid activation function.

```markdown
## Contribution Metadata
Author(s): Emily Jenks
Section: ResNet18 part 2
Method: [resnet18_transfer_learning, random_search, hyperparameter_tuning, model_selection]
SHORT Description of contribution:
In this section, I implemented hyperparameter tuning for a pretrained ResNet18 model using random search to
identify an optimal configuration.
I defined a search space including learning rate, optimizer, batch size, dropout, and transfer learning
freezing strategies, and randomly sampled combinations to evaluate on the validation set. Each configuration
was trained and evaluated using metrics such as AUC, accuracy, and F1-score, and the results were compared
to select the best-performing model based on validation AUC.

Difficulty: INTERMEDIATE
```
### ResNet18 Model Selection
### Hyperparameter Tuning and Model Selection for ResNet18

After defining ResNet18 as the proposed deeper architecture, we perform hyperparameter tuning in order to select a strong configuration for the final model. This step is important because the performance of a deep learning model does not depend only on the architecture itself, but also on training choices such as the learning rate, optimizer, batch size, dropout, and fine-tuning strategy. In this project, ResNet18 is used as a pretrained transfer learning model, meaning that the network already contains weights learned from a large image dataset and can reuse general visual features such as edges, textures, shapes, and object-level patterns. However, for our specific task of real-vs-AI image classification, we need to decide how much of this pretrained knowledge should remain fixed and how much should be adapted to our dataset. This is controlled through the freezing strategy: we can train only the final classification layer, fine-tune the last residual block, or update the entire network. Random search is valuable here because there is no obvious best combination of learning rate, optimizer, batch size, dropout, and freezing strategy in advance. Instead of manually choosing one configuration, random search samples several possible combinations from the search space and compares them on the validation set. This makes the model selection process more systematic and helps us choose the ResNet18 setup that best supports our project goal: evaluating whether a deeper pretrained model can outperform the shallow CNN baselines when trained under an appropriate configuration.

Random search was applied to ResNet18 rather than to the shallow CNNs because ResNet18 was our proposed final model and had more training choices to optimize, especially due to transfer learning. In particular, the freezing strategy determines how much of the pretrained network is fine-tuned, which can strongly affect performance. The shallow CNNs, instead, were used mainly as controlled baseline models, so we kept their setup simpler in order to compare the effect of model depth without introducing too many additional tuning variables.




In [ ]:
# Hyperparameter tuning for ResNet18 using a random search over key training and transfer learning settings
if train == True:
  # Define the search space for both training parameters and layer freezing options
  search_space_resnet = {
    "lr": [1e-2, 1e-3, 1e-4],
    "optimizer": ["adam", "sgd"],
    "batch_size": [16, 32, 64],
    "dropout": [0.3, 0.5],
    "freeze_strategy": ["fc_only", "layer4", "none"]
  }

  # Generate all combinations, then sample a subset to keep the search efficient
  all_combos_resnet = list(itertools.product(*search_space_resnet.values()))
  random.seed(SEED)
  sampled_combos_resnet = random.sample(all_combos_resnet, k=5)

  # Store parameter names for mapping combinations to dictionaries
  keys_resnet = list(search_space_resnet.keys())

  # Binary classification loss (expects raw logits from the model)
  criterion = nn.BCEWithLogitsLoss()
  search_results_resnet = []

  print("\n===== Random search: ResNet18 pretrained proposed model =====")

  # Iterate through sampled configurations, selecting random combinations of parameters, applying to get best results
  for combo in sampled_combos_resnet:
    config = dict(zip(keys_resnet, combo))
    print(f"\nTrying: {config}")

    # Creates the PyTorch dataset objects for training and validation
    train_ds = ImageDataset(train_data, IMAGE_DIR, train_transform)
    val_ds = ImageDataset(val_data, IMAGE_DIR, eval_transform)
    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False)

    # Builds a ResNet18 model with the selected dropout and freezing strategy
    model = build_resnet18(
        dropout=config["dropout"],
        freeze_strategy=config["freeze_strategy"]
    ).to(DEVICE)

    # Creates the optimizer for the current configuration (only trainable parameters)
    optimizer = get_optimizer(
        model,
        config["optimizer"],
        config["lr"],
        only_trainable=True
    )

    # Trains each candidate configuration for a small number of epochs
    for epoch in range(5):
        train_one_epoch(model, train_loader, optimizer, criterion)

    # Evaluates the trained configuration on the validation set
    val_metrics = evaluate(model, val_loader, criterion)

    # Store key validation metrics for comparison
    config["val_auc"] = round(val_metrics["auc"], 4)
    config["val_acc"] = round(val_metrics["accuracy"], 4)
    config["val_f1"] = round(val_metrics["f1"], 4)
    search_results_resnet.append(config)

    print(
        f"  → Val AUC: {val_metrics['auc']:.4f} | "
        f"Acc: {val_metrics['accuracy']:.4f} | "
        f"F1: {val_metrics['f1']:.4f}"
    )
  # Creating the dataframe of the all the combinations of random search in such a way that the best model, according to their val_AUC value, is at the head
  results_df_resnet = pd.DataFrame(search_results_resnet).sort_values("val_auc", ascending=False)

  # Printing the top 5 combinations of random search
  print("\nTop configurations (ResNet18):")
  print(results_df_resnet.head(5).to_string(index=False))

  # Selects the configuration with the highest validation AUC
  best_config_resnet = results_df_resnet.iloc[0].to_dict() #saving the best config combination (result at the head)
  print(f"\nBest ResNet18 config: {best_config_resnet}")


The code above implements this process by first defining `search_space_resnet`, which contains the hyperparameters tested for the ResNet18 model. The learning rate `lr` controls the size of the weight updates during optimization, while `optimizer` selects the update algorithm, either Adam or SGD. The `batch_size` determines how many images are processed before each optimization step, and `dropout` controls the amount of regularization applied before the final classifier by randomly deactivating neurons during training, reducing the risk of overfitting. The `freeze_strategy` further helps prevent overfitting by keeping the pretrained convolutional weights fixed, which stops the model from overwriting general visual features learned on ImageNet with patterns that may be too specific to our training data. The `freeze_strategy` parameter is specific to transfer learning and determines which parts of the pretrained ResNet18 are trainable: `fc_only` trains only the final classification head, `layer4` fine-tunes the last residual block together with the classifier, and `none` leaves the whole network trainable. The code uses `itertools.product()` to generate all possible combinations of these values, then applies `random.sample()` to select five configurations, using `random.seed(SEED)` to make the random search reproducible. For each sampled combination, the tuple of values is converted into a dictionary called `config`, making it easier to access the selected hyperparameters by name. The code then creates `ImageDataset` objects for training and validation, and wraps them into `DataLoader`s using the current batch size. The training loader uses `shuffle=True` so that the model sees the training examples in a different order, while the validation loader uses `shuffle=False` to keep evaluation deterministic. A ResNet18 model is built using `build_resnet18()`, passing the selected `dropout` and `freeze_strategy`, and the optimizer is created using `get_optimizer()`, with `only_trainable=True` so that frozen layers are not updated. Each configuration is trained for five epochs using `train_one_epoch()`, then evaluated on the validation set using `evaluate()`. The validation metrics stored are AUC, accuracy, and F1-score. Finally, the results are stored in `results_df_resnet`, sorted by `val_auc` in descending order, and the best configuration is selected as the one with the highest validation AUC.


```markdown
## Contribution Metadata
Author(s): Ludovica Tapparo
Section: ResNet18 part 3
Method: [transfer_learning, best_configuration, pytorch_dataloader, final_training, model_saving]
SHORT Description of contribution: In this section, I contributed by
developing and explaining the final ResNet18 training and saving pipeline. I described how
the best configuration is used to train the model on the combined training
and validation data, and how saving the model weights and search results supports
reproducibility and later reuse.


Difficulty: INTERMEDIATE
```
### ResNet18 Full training

Following hyperparameter optimization, the ResNet18 model is retrained using the best-performing configuration found during random search. To maximize the amount of data available for learning, the training and validation datasets are combined into a single training set, while the test set remains completely unseen and is reserved for final evaluation.

Therefore, a new ResNet18 model is initialized with the optimal hyperparameters, including the selected dropout rate, learning rate, optimizer, batch size, and layer-freezing strategy. Depending on the chosen freeze strategy, either part or all of the pretrained network weights are updated during training. The model leverages weights pretrained on ImageNet, enabling transfer learning from large-scale image classification tasks to the deepfake detection domain.

The combined dataset is loaded using a PyTorch DataLoader with shuffling enabled to ensure that training samples are presented in a random order during each epoch. The optimizer is then configured using the selected hyperparameters and applied only to trainable parameters within the network.

The model is trained for 10 epochs on the full dataset. During each epoch, the training loop performs forward propagation, computes the classification loss, backpropagates gradients, and updates the network weights. The training loss is recorded and displayed after every epoch to monitor convergence. The relatively small number of epochs is sufficient because the pretrained ResNet18 architecture converges quickly on the task, benefiting from transfer learning and the optimized learning rate discovered during hyperparameter tuning.

After training is complete, the final model weights are saved to disk along with the hyperparameter search results.

In [ ]:
#if training, the code will run
if train == True:
  full_train = pd.concat([train_data, val_data]).reset_index(drop=True) #combine training data set and validation dataset

  full_ds_resnet = ImageDataset(full_train, IMAGE_DIR, train_transform) #store images by calling the function ImageDataset helper function

  #Data Routing Loaders
  #format the input for the model in a way that the model will understand
  full_loader_resnet = DataLoader(
    full_ds_resnet,
    batch_size=int(best_config_resnet["batch_size"]),
    shuffle=True
  )

  #build resnet model on best config parameters
  best_model_resnet = build_resnet18(
    dropout=float(best_config_resnet["dropout"]),
    freeze_strategy=best_config_resnet["freeze_strategy"]
  ).to(DEVICE)

#getting the best optimizer by calling the get_optimizer helper function
  optimizer = get_optimizer(
    best_model_resnet,
    best_config_resnet["optimizer"],
    float(best_config_resnet["lr"]),
    only_trainable=True
  )
#train the model with a set number of epochs
  EPOCHS = 10
  for epoch in range(EPOCHS):
    loss = train_one_epoch(best_model_resnet, full_loader_resnet, optimizer, criterion)
    print(f"Epoch {epoch + 1}/{EPOCHS} — loss: {loss:.4f}")


After selecting the best ResNet18 configuration through validation-based model selection, the final model is trained using both the original training set and the validation set. This is done only when `train == True`, so the training process is executed only when needed. Since the validation set has already served its purpose during hyperparameter tuning, it is merged with the training set using `pd.concat()` to allow the final model to learn from all available labelled data before testing.


In [ ]:
#Saving the model and the results of best config
if train == True:
  os.makedirs("/content/resnet", exist_ok=True) #make the folder
  results_df_resnet.to_csv(f"/content/resnet/resnet_search_results.csv", index=False) #saving the results of best config into CSV
  torch.save(best_model_resnet.state_dict(), f"/content/resnet/resnet18.pth") #saving the best model

  !zip -r /content/resnet.zip {"/content/resnet"} #zipping the folder

  #downloading the zipfile
  print("Downloading zip file to your local computer...")
  files.download('/content/resnet.zip')

This cell saves the final ResNet18 model and the hyperparameter search results to make the experiment reproducible and reusable. The code is executed only when `train == True`, meaning that files are saved only after the model has actually been trained. First, `os.makedirs()` creates a dedicated folder called `/content/resnet`, where the outputs of the ResNet18 experiment are stored. The DataFrame `results_df_resnet`, which contains the tested random search configurations and their validation metrics, is saved as a CSV file. This allows us to keep a record of the model selection process and verify which configuration was selected as the best one.

The trained model weights are then saved using `torch.save(best_model_resnet.state_dict(), ...)`. Instead of saving the entire model object, the code saves the model’s `state_dict`, which contains the learned parameters of the network. This is the standard PyTorch approach because the model architecture can be rebuilt later and the saved weights can be loaded into it. Finally, the folder is compressed into a `.zip` file and downloaded from the Colab environment. This makes it possible to preserve the trained model and search results outside the temporary notebook runtime, allowing the experiment to be reloaded, shared, or evaluated again without retraining the model from scratch.


```markdown
## Contribution Metadata
Author(s): Chiara Bosi
Section: ResNet18 part 4
Method: [model_loading, final_evaluation, confusion_matrix]
SHORT Description of contribution:
I contributed to the final evaluation of the ResNet18 model.
This includes loading the pre-trained weights and best hyperparameter configuration, setting up the test pipeline and reporting the final test metrics together with the confusion matrix.


Difficulty: INTERMEDIATE
```
### ResNet18 final evaluation

Once training is complete, the model needs to be loaded and evaluated on the held-out test set.

When `train` is set to `False`, the training loop is bypassed entirely. Rather than re-running the full pipeline, the pre-trained ResNet18 archive is downloaded from Google Drive using `gdown` and its unique file ID.

The archive (`resnet.zip`) contains our best model weights and the optimal hyperparameter combination found during random search. Once downloaded, the file is extracted locally into the `/content/resnet` directory to be ready for immediate testing.

In [ ]:
if train == False:
  # Download the pre-saved ResNet18 archive from Google Drive.
  # The zip contains both the trained weights and the random search results CSV.

  RESNET_ZIP_FILE_ID = "1XYa1PAw2ZJc8r5zIy3RJYTLCvWxi6oRn"
  RESNET_ZIP_DRIVE_SHARE_LINK = f"https://drive.google.com/uc?id={RESNET_ZIP_FILE_ID}"
  RESNET_ZIP_PATH = "/content/resnet.zip"
  RESNET_WEIGHTS_LOCATION = "/content/resnet"

  gdown.download(RESNET_ZIP_DRIVE_SHARE_LINK, RESNET_ZIP_PATH, quiet=False)
  !unzip -o {RESNET_ZIP_PATH} -d /

To reconstruct the model in the exact same configuration it was trained in, the random search results CSV is loaded and the best-performing configuration is converted to a dictionary (`best_config_resnet`).

This contains all the hyperparameters needed to correctly rebuild the pipeline: the optimal batch size for the test loader, the dropout rate and freeze strategy for `build_resnet18()`.
Once the architecture is instantiated with these values, the saved weights (`resnet18.pth`) are loaded into the model so that it is ready for evaluation.

In [ ]:
# Load the random search results and extract the best configuration.
results_df_resnet = pd.read_csv(f"/content/resnet/resnet_search_results.csv")
best_config_resnet = results_df_resnet.iloc[0].to_dict()

criterion = nn.BCEWithLogitsLoss()

# Build the test dataset and loader using the eval transform
test_ds_resnet = ImageDataset(test_data, IMAGE_DIR, eval_transform)
test_loader_resnet = DataLoader(
    test_ds_resnet,
    batch_size=int(best_config_resnet["batch_size"]),
    shuffle=False
  )

# Rebuild the ResNet18 architecture with the best dropout and freeze strategy, then load the saved weights.
best_model_resnet = build_resnet18(
    dropout=float(best_config_resnet["dropout"]),
    freeze_strategy=best_config_resnet["freeze_strategy"]
  ).to(DEVICE)
saved_state_dict = torch.load(f'/content/resnet/resnet18.pth', map_location=DEVICE)
best_model_resnet.load_state_dict(saved_state_dict)

The final model is passed to the `evaluate` helper function together with the test loader and the loss criterion. This returns the four key performance metrics — **loss, accuracy, AUC-ROC and F1-score** — which are printed below.

A confusion matrix is then plotted using `plot_confusion_matrix` to better visualize the classification errors across the two classes.

Finally, the final model state is saved as `proposed_resnet18.pth`.

In [ ]:
# Run evaluation on the test set and collect the four key metrics.
test_metrics_resnet = evaluate(best_model_resnet, test_loader_resnet, criterion)

print("\n=== PROPOSED MODEL TEST RESULTS: RESNET18 ===")
print(f"Accuracy  : {test_metrics_resnet['accuracy']:.4f}")
print(f"AUC-ROC   : {test_metrics_resnet['auc']:.4f}")
print(f"F1-score  : {test_metrics_resnet['f1']:.4f}")

# Plot the confusion matrix to visualize classification errors per class.
plot_confusion_matrix(
    best_model_resnet,
    test_loader_resnet,
    title="Proposed ResNet18 — Confusion Matrix",
    cmap="Blues"
)

# Save the final model weights for reproducibility.
torch.save(best_model_resnet.state_dict(), "proposed_resnet18.pth")
print("Proposed ResNet18 model saved.")


The ResNet18 achieved an accuracy of 0.9927, an AUC-ROC of 0.9997 and an F1-score of 0.9926, outperforming all shallow CNN baselines across every metric.

This result was expected: a pretrained deep architecture has a significant advantage on this task, since detecting AI-generated images often relies on subtle texture-level artifacts that shallow networks struggle to capture without a much larger training set. The performance gap confirms that transfer learning was a well-suited choice for this problem.

## 5. Explainability

```markdown
## Contribution Metadata
Author(s): Ludovica Tapparo
Section: Explainability part 1
Method: [lime_explainability]
SHORT Description of contribution:
My contribution to this section was to develop the code for the application of LIME
and explain the role of LIME as an interpretability method for the real-vs-AI image
classifier. I described how the image is divided into superpixels, how the model
predictions are converted into class probabilities suitable for LIME, and how the
resulting attribution map highlights the regions that support the model’s prediction.


Difficulty: INTERMEDIATE
```

### LIME Explainability

Deep learning models can achieve strong performance in image classification, but their predictions are often difficult to interpret because the decision process is distributed across many layers and learned feature representations. For this reason, we include an explainability step using LIME, which stands for Local Interpretable Model-agnostic Explanations. LIME is a post-hoc explainability method, meaning that it is applied after the model has already been trained. It is model-agnostic, meaning that it is not structured for a specific class of models, but rather it can shed light on the decision process of any model. Its goal is not to explain the full internal structure of the neural network, but to explain a single prediction locally. In the case of images, LIME divides the image into interpretable regions called superpixels, then repeatedly perturbs the image by hiding or modifying some of these regions. By observing how the model’s predicted probability changes when different superpixels are removed, LIME estimates which regions contributed most to the prediction. This is useful in our project because the task of distinguishing real images from AI-generated images may depend on subtle visual patterns, such as texture inconsistencies, unnatural details, or local artifacts. LIME allows us to inspect whether the model is focusing on meaningful parts of the image instead of relying on irrelevant background regions or spurious correlations.



In [ ]:
CLASS_NAMES = ["AI-generated", "Real"]


def get_lime_preprocess():
    """Returns the same preprocessing pipeline used during model evaluation.
    LIME explanations should be generated on inputs prepared in the same way
    as the images seen by the trained model"""
    return eval_transform


def inverse_normalize_tensor(tensor):
    """Reverses the ImageNet normalization applied during preprocessing.
    This is useful when we want to display the image in its original color range"""
    mean = torch.tensor([0.485, 0.456, 0.406], device=tensor.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=tensor.device).view(3, 1, 1)

    # Brings pixel values back to the [0, 1] range for visualization
    return (tensor * std + mean).clamp(0, 1)


def load_project_image_and_segment(image_path, n_segments=50, compactness=10):
    """Load an image, prepare it for the model, and compute its SLIC superpixels.
    The superpixels are the interpretable regions used by LIME."""

    # Load the image in RGB format and resize it to the input size used by the models
    original_image = Image.open(image_path).convert("RGB").resize(IMG_SIZE)
    # Apply the evaluation transform and add a batch dimension
    input_tensor = get_lime_preprocess()(original_image).unsqueeze(0)

    # Segment the visible RGB image into superpixels.
    # LIME will perturb these regions instead of individual pixels.
    segments = slic(
        np.array(original_image),
        n_segments=n_segments,
        compactness=compactness,
        start_label=0
    )

    return original_image, input_tensor, segments


def batch_predict_binary_for_lime(images, model):
    """
    Return class probabilities in the format required by Captum LIME.
    The model outputs one logit for binary classification.
    We convert it into probabilities for both classes:
    [P(AI-generated), P(Real)].
    """
    model.eval()
    images = images.to(DEVICE)

    with torch.no_grad():
        # Convert the raw binary logit into the probability of the Real class.
        logits = model(images).squeeze(1)
        prob_real = torch.sigmoid(logits)
        # Since the task is binary, the other probability is its complement.
        prob_ai = 1.0 - prob_real
        # Captum expects one probability column per class.
        probs = torch.stack([prob_ai, prob_real], dim=1)

    return probs


def predict_project_image(model, input_tensor):
    """Predict the class of one image and return both the predicted index
    and the probability assigned to each class."""
    model.eval()
    with torch.no_grad():
        probs = batch_predict_binary_for_lime(input_tensor, model).cpu().numpy()[0]
    # Select the class with the highest probability.
    pred_class = int(np.argmax(probs))
    return pred_class, probs


def explain_with_captum_lime_binary(model, input_tensor, segments, target_class, n_samples=500):
    """Generate a LIME explanation for the selected target class.
    The returned attributions indicate how much each region supports the prediction."""

    # Perturbed samples closer to the original image are weighted more strongly.
    similarity_func = get_exp_kernel_similarity_function(
        distance_mode="cosine",
        kernel_width=1.0
    )
    # LIME uses a simple local model to approximate the neural network near this image.
    interpretable_model = SkLearnLinearRegression()

    # Create the Captum LIME explainer using our probability function.
    lime = Lime(
        forward_func=lambda x: batch_predict_binary_for_lime(x, model),
        interpretable_model=interpretable_model,
        similarity_func=similarity_func
    )

    # The feature mask assigns each pixel to a superpixel region.
    # Shape (1, 1, H, W) applies the same region mask to all RGB channels.
    feature_mask = torch.tensor(segments, dtype=torch.long).unsqueeze(0).unsqueeze(0).to(DEVICE)

    # Estimate the contribution of each superpixel by perturbing the image regions.
    attributions = lime.attribute(
        inputs=input_tensor.to(DEVICE),
        target=target_class,
        n_samples=n_samples,
        perturbations_per_eval=32,
        feature_mask=feature_mask
    )

    return attributions


def visualize_lime_explanation(attributions, original_image, segments=None, title="LIME explanation"):
    """Overlay the positive LIME attributions on the original image.
    Positive regions are those that support the selected class."""
    attr = attributions.squeeze().detach().cpu().numpy()

    # If attributions are returned per RGB channel, average them into one heatmap.
    if attr.ndim == 3 and attr.shape[0] == 3:
        attr = np.mean(attr, axis=0)
    # Add a channel dimension if the attribution map is 2D.
    if attr.ndim == 2:
        attr = np.expand_dims(attr, axis=2)
    # Prepare the original image in [0, 1] format for plotting.
    original_np = np.array(original_image.resize(IMG_SIZE)) / 255.0

    _ = visualize_image_attr(
        attr,
        original_image=original_np,
        method="blended_heat_map",
        sign="positive",
        show_colorbar=True,
        title=title
    )

     # Optionally show the superpixel boundaries used to build the explanation.
    if segments is not None:
        plt.figure(figsize=(5, 5))
        plt.imshow(mark_boundaries(original_np, segments))
        plt.title("SLIC superpixels used by LIME")
        plt.axis("off")
        plt.show()


LIME is implemented with Captum , which is a Pytorch interpretability library. The code implements a LIME pipeline adapted to our binary classification setting. The class names are defined as `["AI-generated", "Real"]`, matching the two possible outputs of the model. The function `get_lime_preprocess()` returns the same evaluation transform used by the trained models, ensuring that the image explained by LIME is preprocessed consistently with the images used during validation and testing. The function `inverse_normalize_tensor()` reverses the ImageNet normalization by applying the original mean and standard deviation values, which is useful when visualizing tensors in a natural color range. The function `load_project_image_and_segment()` loads a selected image, converts it to RGB, resizes it to the model input size, applies the evaluation preprocessing, and then uses SLIC segmentation to divide the image into superpixels. These superpixels are important because LIME explains predictions in terms of image regions rather than individual pixels, making the explanation more interpretable.

The function `batch_predict_binary_for_lime()` adapts the output of our binary model to the format expected by Captum LIME. Our model returns a single raw logit, which is converted into the probability of the `Real` class using the sigmoid function. Since this is a binary task, the probability of the `AI-generated` class is computed as `1 - P(Real)`. These two probabilities are then stacked into the format `[P(AI-generated), P(Real)]`, allowing LIME to treat the problem as a two-class classification task. The function `predict_project_image()` uses this probability function to obtain the predicted class and its class probabilities for a single image. The main explanation function, `explain_with_captum_lime_binary()`, creates a Captum `Lime` object using the model prediction function, a local interpretable linear regression model, and a similarity function that gives higher importance to perturbed samples closer to the original image. It then builds a `feature_mask` from the SLIC superpixels, so that LIME perturbs meaningful image regions rather than individual pixels. Finally, `visualize_lime_explanation()` converts the resulting attributions into a heatmap and overlays the positive contributions on the original image. The optional superpixel boundary visualization also shows the regions used by LIME when generating the explanation.


```markdown
## Contribution Metadata
Author(s): Emily Jenks
Section: Explainability part 2
Method: [superpixels, feature attribution, image segmentation]
SHORT Description of contribution:
In this section, I applied LIME (Local Interpretable Model-Agnostic Explanations) to visualize
how the trained ResNet18 model makes predictions on test images. I selected one AI-generated
and one real image and generated explanations by segmenting the images into superpixels and
identifying which regions most influenced the model’s decision. The results were visualized to
highlight important areas contributing to each classification.

Difficulty: INTERMEDIATE
```
### LIME visualization

This code selects representative examples for LIME visualization by sampling one AI-generated image and one real image from the test dataset. It filters the dataset based on binary labels, randomly selects one image per class, and stores metadata such as file path and label. The selected samples are organized into a DataFrame for inspection. This ensures that the explainability analysis is performed on balanced and interpretable examples from both classes.

In [ ]:
# Select one AI-generated and one real image from the held-out test set.
# binary_label: 0 = AI-generated, 1 = Real.
lime_examples = [] # List to store selected examples for LIME visualization

# Loop through both classes: AI-generated (0) and Real (1)
for label_value, label_name in [(0, "AI-generated"), (1, "Real")]:
    # Filter test dataset to only include images of the current class
    subset = test_data[test_data["binary_label"] == label_value]

    # Ensure there is at least one image in the subset
    if len(subset) > 0:
        row = subset.sample(1, random_state=SEED).iloc[0] # Randomly sample one image (reproducible with SEED)
        image_path = os.path.join(IMAGE_DIR, row["filename"]) # Construct full file path to the image

        # Store relevant information for later LIME analysis
        lime_examples.append({
            "true_label": label_name,
            "true_label_id": label_value,
            "filename": row["filename"],
            "image_path": image_path
        })

# Convert selected examples into a DataFrame for easy viewing
lime_examples_df = pd.DataFrame(lime_examples)
display(lime_examples_df[["filename", "true_label"]])

This code generates LIME explanations for selected test images using the trained ResNet18 model. Each image is first segmented into superpixels, which serve as interpretable regions for perturbation. The model predicts class probabilities, and LIME (via Captum) approximates the model’s behavior by sampling perturbations of these segments. The resulting attributions indicate which regions most influenced the prediction. Finally, the explanation is visualized as an overlay on the original image, enabling intuitive interpretation of model decision-making.

In [ ]:
# Run LIME explanations on the selected test images using the proposed ResNet18 model.
# n_samples controls the trade-off between explanation quality and runtime
# (200 = fast, 500 = balanced, 1000 = higher quality but slower)

LIME_N_SAMPLES = 500

# Loop through each selected example (AI and Real)
for example in lime_examples:
    print("\n" + "=" * 80)
    print(f"Image: {example['filename']}")
    print(f"True label: {example['true_label']}")

    # Load image, preprocess it, and generate superpixel segmentation
    original_image, input_tensor, segments = load_project_image_and_segment(
        example["image_path"],
        n_segments=50, # Number of superpixels
        compactness=10 # Controls shape/size of segments
    )

    # Get model prediction and class probabilities
    pred_class, probs = predict_project_image(best_model_resnet, input_tensor)

    print(f"Predicted class: {CLASS_NAMES[pred_class]}")
    print(f"P(AI-generated): {probs[0]:.4f}")
    print(f"P(Real):         {probs[1]:.4f}")

    # Compute LIME attributions using Captum
    attributions = explain_with_captum_lime_binary(
        model=best_model_resnet,
        input_tensor=input_tensor,
        segments=segments,
        target_class=pred_class,
        n_samples=LIME_N_SAMPLES
    )

    # Visualize the explanation overlay on the image
    visualize_lime_explanation(
        attributions,
        original_image,
        segments=segments,
        title=f"LIME explanation — predicted: {CLASS_NAMES[pred_class]}"
    )


The region-based feature attribution maps generated by the LIME framework provide structural insights into how the proposed ResNet18 model groups image segments to form its classification decisions. Rather than analyzing isolated pixels, LIME utilizes the SLIC (Simple Linear Iterative Clustering) algorithm to partition each test image into distinct, structurally uniform patches called superpixels, which are clearly delineated by the yellow boundary lines. The model then perturbs these individual bounded segments to calculate their specific feature weightings, mapping the most influential regions in shades of green. The presence of this green overlay represents areas that positively contributed to the model’s predicted class, meaning these specific superpixels increased the model’s confidence in its decision, with darker or more concentrated green regions indicating a stronger strategic influence.

When evaluating images from the test set, the model does not rely on the entire frame uniformly but instead isolates specific localized structures. For images predicted as AI-generated, the green regions typically cluster around yellow-bounded superpixels flanking critical blending junctions—such as segments covering hair textures, jawline perimeters, or the transition spaces where the subject meets the background. This demonstrates that when detecting a synthetic face, the network actively audits localized regions where generative artifacts, edge unnaturalness, or structural blending inconsistencies are highly likely to manifest.

Conversely, when processing images predicted as Real, the green patches shift to emphasize organic, fine-grained details, often locking onto yellow superpixel cells that map core facial geometry like the eyes, nasal ridge, mouth, or central facial symmetry. By highlighting these structurally important parts of the face, the network validates the natural texturing and geometric proportions characteristic of genuine photography. Ultimately, seeing the green overlays align with these intuitive, visually meaningful structures validates the model’s behavior, providing transparency and proving that the ResNet18 classifier relies on stable biometric regions rather than irrelevant background noise or dataset bias.

```markdown
## Contribution Metadata
Author(s): Chiara Bosi
Section: Explainability part 3
Method: [integrated_gradients_explainability]
SHORT Description of contribution:
In this part of the project, I contributed to the implementation and analysis of gradient-based explainability techniques for the proposed ResNet18 classifier.
In particular, I focused on understanding  why Integrated Gradients is well suited for this task, how attribution heatmaps can help identify visually meaningful patterns, as well as interpreting pixel-level attribution patterns and comparing gradient-based explanations with perturbation-based approaches such as LIME.

Difficulty: INTERMEDIATE
```


### **Gradient-Based Explainability: Integrated Gradients**

While LIME explains predictions by perturbing regions of the image and observing how the output changes, Integrated Gradients relies directly on the internal gradient structure of the model.
Specifically, this approach computes feature attributions by accumulating gradients along a straight interpolation path from a baseline image, in our case a black image, to the actual input. Intuitively, this measures how much each pixel contributed to moving the model's output from the baseline prediction to the prediction on the real input. The result is an attribution map that highlights regions of the image most relevant to the predicted class.

Compared to LIME, Integrated Gradients produces smoother attribution maps because it operates at the pixel level rather than grouping pixels into superpixels.
Therefore, this method is particularly useful for identifying subtle visual patterns that strongly influence the classifier's prediction, such as texture irregularities, facial details, or small synthetic artifacts.

Finally, we decided to apply Integrated Gradients to the same images previously analyzed with LIME in order to compare whether both explainability approaches consistently target similar image regions.

In [ ]:
def forward_binary_logit(model, inputs):
    """
    Forward function for Captum Integrated Gradients.

    The model outputs a single binary logit:
    - positive values indicate stronger evidence for the Real class
    - negative values indicate stronger evidence for the AI-generated class

    To explain the AI-generated class, we use the negative logit.
    To explain the Real class, we use the original logit.
    """
    return model(inputs).squeeze(1)


def explain_with_integrated_gradients(
    model,
    input_tensor,
    target_class,
    n_steps=50
):
    """
    Compute Integrated Gradients attributions for a binary image classifier.

    target_class:
    - 0 = AI-generated
    - 1 = Real
    """
    model.eval()

    integrated_gradients = IntegratedGradients(
        lambda x: forward_binary_logit(model, x)
        if target_class == 1
        else -forward_binary_logit(model, x)
    )

    baseline = torch.zeros_like(input_tensor).to(DEVICE)

    attributions = integrated_gradients.attribute(
        input_tensor.to(DEVICE),
        baselines=baseline,
        n_steps=n_steps
    )

    return attributions

The implementation uses Captum’s `IntegratedGradients` module to compute attribution maps for the proposed ResNet18 classifier.

Since the model outputs a single binary logit, the helper function `forward_binary_logit()` defines how the prediction should be interpreted for each target class:
- positive logits correspond to the Real class;
- negative logits correspond to the AI-generated class.

The `explain_with_integrated_gradients()` function then computes the attribution scores by integrating the gradients between a baseline image and the original input image over multiple interpolation steps.

In [ ]:
def visualize_integrated_gradients(
    attributions,
    original_image,
    title="Integrated Gradients explanation"
):
    """
    Visualize Integrated Gradients as a heatmap over the original image.
    """
    attr = attributions.squeeze().detach().cpu().numpy()

    # Convert tensor to image format
    attr = np.transpose(attr, (1, 2, 0))

    original_np = np.array(original_image.resize(IMG_SIZE)) / 255.0

    _ = visualize_image_attr(
        attr,
        original_image=original_np,
        method="blended_heat_map",
        sign="absolute_value",
        show_colorbar=True,
        title=title
    )

```markdown
## Contribution Metadata
Author(s): Vatsalya Arora
Section: Explainability part 4
Method: [explainable_ai, gradient_attribution, perturbation_vs_gradient, model_confidence, artifact_verification]
SHORT Description of contribution:
This section implements Integrated Gradients (IG), a gradient-based Explainable AI method, to analyze what our proposed ResNet18 model is focusing on.
For each test sample, the code prints the true versus predicted class probabilities,
computes pixel-level attribution scores over 50 steps (IG_N_STEPS), and renders a visual attribution map.
Looking at the generated logs for images the model shows extreme confidence ($100\%$ probability scores) for both classes.
The text highlights a crucial architectural comparison: while LIME relies on coarse, superpixel-based perturbations,
Integrated Gradients provides fine-grained, pixel-level attributions.
The fact that both approaches point to the same structural areas (like facial boundaries and textures)
confirms that our network is recognizing genuine visual patterns rather than just memorizing trivial dataset background artifacts.
Difficulty: INTERMEDIATE
```
### Integrated gradients visualization

The resulting attribution maps are visualized as blended heatmaps superimposed on the original image. Regions with stronger activations correspond to image areas that contribute more significantly to the predicted class according to the internal gradient flow of the proposed ResNet18 model.

In [ ]:
IG_N_STEPS = 50

for example in lime_examples:
    print("\n" + "=" * 80)
    print(f"Image: {example['filename']}")
    print(f"True label: {example['true_label']}")

    original_image, input_tensor, _ = load_project_image_and_segment(
        example["image_path"],
        n_segments=50,
        compactness=10
    )

    pred_class, probs = predict_project_image(best_model_resnet, input_tensor)

    print(f"Predicted class: {CLASS_NAMES[pred_class]}")
    print(f"P(AI-generated): {probs[0]:.4f}")
    print(f"P(Real):         {probs[1]:.4f}")

    ig_attributions = explain_with_integrated_gradients(
        model=best_model_resnet,
        input_tensor=input_tensor,
        target_class=pred_class,
        n_steps=IG_N_STEPS
    )

    visualize_integrated_gradients(
        ig_attributions,
        original_image,
        title=f"Integrated Gradients — predicted: {CLASS_NAMES[pred_class]}"
    )

The pixel-level feature attribution maps generated by the Integrated Gradients (IG) framework highlight the precise visual structures the proposed ResNet18 model utilizes to differentiate between synthetic and authentic faces. When evaluating images classified as AI-generated, the blue attribution density typically concentrates heavily along structural blending transitions—such as the perimeter boundaries of the hair, jawline and background context—as well as localized micro-textures across the central facial mask. This focus demonstrates that the network is actively auditing the model for synthesis anomalies, edge-smoothing artifacts, and surface consistency errors where the deepfake features merge with the environment.

In contrast, when evaluating samples predicted as Real, the model shifts its high-density pixel attributions directly toward sharp, natural anatomical features, heavily tracing core geometric landmarks like the contours of the eyes, eyebrows, and nasal ridge. The widespread, structurally coherent coverage across these key facial zones indicates that the network is validating the natural geometric proportions and high-frequency edge details inherent to authentic photography. Ultimately, this gradient-based diagnostic confirms that the ResNet18 classifier successfully bypasses trivial background shortcuts or random dataset noise, consistently relying on visually meaningful biometric regions to make its final decisions regardless of the specific test image processed.

LIME and Integrated Gradients provide complementary perspectives on the proposed model behavior.

Despite relying on different explainability paradigms, both methods frequently highlight similar semantic regions across the analyzed test images. In many examples, the classifier focuses on facial details, texture inconsistencies and background irregularities when detecting AI-generated images.

At the same time, the two approaches differ in the structure of their explanations. As mentioned before, LIME produces region-based interpretations constrained by the superpixel segmentation, whereas Integrated Gradients generates smoother attribution maps that capture more fine-grained visual patterns.

The consistency between perturbation-based and gradient-based explanations suggests that ResNet18  is not relying exclusively on random noise or trivial dataset artifacts, but is instead learning visually meaningful patterns.

Although the highlighted regions are often consistent across methods, the generated explanations should still be interpreted as qualitative diagnostic tools rather than exact representations of the model's reasoning process.

## 6. Experimental results

```markdown
## Contribution Metadata
Author(s): Ludovica Tapparo
Section:  
Method: [model_comparison, performance_visualization, depth-performance_analysis]
SHORT Description of contribution: In this section, I contributed by creating and
explaining the final result tables and plots comparing the shallow CNN baselines with
the proposed ResNet18 model. I focused on visualizing accuracy, AUC-ROC, F1-score,
parameter counts, and the performance trend with increasing model depth, in order to
support the project’s conclusion about the benefit of deeper architectures.


Difficulty: BASIC
```
Having come to an end with our experimental tests, we now visualize the most salient results.

### Final model comparison

In [ ]:
# Collect the final test results of the two shallow CNN baselines.
final_results = cnn_test_results.copy()

# Add the proposed ResNet18 model to the same table.
resnet_row = {
    "model": "ResNet18-Pretrained",
    "num_blocks": 18,
    "architecture": "Pretrained ResNet18",
    "test_accuracy": test_metrics_resnet["accuracy"],
    "test_auc": test_metrics_resnet["auc"],
    "test_f1": test_metrics_resnet["f1"]
}
# Combine the CNN baseline results and the ResNet18 result in one table.
final_results = pd.concat(
    [final_results, pd.DataFrame([resnet_row])],
    ignore_index=True
)
# Count both the total number of parameters and the trainable parameters.
# This helps compare model complexity.
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

cnn_small_params = count_parameters(final_cnn_models["CNN-Small-2ConvBlocks"])
cnn_medium_params = count_parameters(final_cnn_models["CNN-Medium-3ConvBlocks"])
resnet_params = count_parameters(best_model_resnet)

param_info = {
    "CNN-Small-2ConvBlocks": cnn_small_params,
    "CNN-Medium-3ConvBlocks": cnn_medium_params,
    "ResNet18-Pretrained": resnet_params
}
# Add total and trainable parameter counts to the comparison table.
final_results["total_params"] = final_results["model"].map(lambda m: param_info[m][0])
final_results["trainable_params"] = final_results["model"].map(lambda m: param_info[m][1])

final_results[["test_accuracy", "test_auc", "test_f1"]] = final_results[
    ["test_accuracy", "test_auc", "test_f1"]
].round(4)

print("Final Test Results: Shallow CNNs vs Proposed ResNet18")
display(final_results[
    ["model", "architecture", "test_accuracy", "test_auc", "test_f1", "total_params", "trainable_params"]
])

This cell creates a final comparison table for the experimental results. It starts from the test results of the two shallow CNN baselines and then adds the final ResNet18 test performance, allowing all models to be compared in a single table. The table reports three evaluation metrics: accuracy, AUC, and F1-score. Accuracy measures the overall proportion of correct predictions, AUC evaluates how well the model separates real and AI-generated images across different decision thresholds, and F1-score balances precision and recall.

In addition to performance, the code also compares model complexity by counting the total number of parameters and the number of trainable parameters for each model. This is done through the `count_parameters()` function, which sums all parameters in the model and separately counts only those with `requires_grad=True`. Including parameter counts is useful because it shows the trade-off between model size and predictive performance. In this way, the table does not only show which model performs best, but also how much additional capacity is introduced when moving from the shallow CNN baselines to the deeper pretrained ResNet18 model.


### Final test metric comparison

In [ ]:
metrics_to_plot = ["test_accuracy", "test_auc", "test_f1"]
metric_labels = ["Accuracy", "AUC-ROC", "F1-score"]

x = np.arange(len(final_results))
width = 0.25

plt.figure(figsize=(10, 6))
for i, metric in enumerate(metrics_to_plot):
    plt.bar(
        x + i * width,
        final_results[metric],
        width,
        label=metric_labels[i])

plt.xticks(
    x + width,
    final_results["model"],
    rotation=20,
    ha="right")
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Final Test Performance Comparison")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

This plot visualizes the final test performance of all models using a grouped bar chart. For each model, the chart reports accuracy, AUC-ROC, and F1-score side by side, making it easier to compare the shallow CNN baselines with the proposed ResNet18 model. Displaying the three metrics together gives a more complete view of model performance than relying on a single score.


### Model depth increase

In [ ]:
# Creates a copy of the final results table.
depth_results = final_results.copy()

# Depth indicator:
# 2 and 3 represent the number of convolutional blocks in CNN-Small and CNN-Medium.
# 18 represents ResNet18.
depth_results["depth_indicator"] = [2, 3, 18]


def plot_metric_vs_depth(df, metric_column, metric_label):
    """
    Plots one performance metric as a function of the model depth indicator.
    """
    plt.plot(
        df["depth_indicator"],
        df[metric_column],
        marker="o",
        label=metric_label)

plt.figure(figsize=(9, 6))

# Plot the three final test metrics.
plot_metric_vs_depth(depth_results, "test_accuracy", "Accuracy")
plot_metric_vs_depth(depth_results, "test_auc", "AUC-ROC")
plot_metric_vs_depth(depth_results, "test_f1", "F1-score")

# Add model names near the AUC curve to identify each point.
for _, row in depth_results.iterrows():
    plt.text(
        row["depth_indicator"],
        row["test_auc"] + 0.01,
        row["model"],
        ha="center",
        fontsize=9)

plt.xlabel("Model depth indicator")
plt.ylabel("Test score")
plt.title("Performance Trend as Model Depth Increases")
plt.ylim(0, 1.05)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

This plot shows how the final test performance changes as the model depth increases from `CNN-Small` to `CNN-Medium` and then to the proposed ResNet18 model. The `depth_indicator` is used as a visual reference: values 2 and 3 correspond to the number of convolutional blocks in the shallow CNNs, while 18 represents the deeper ResNet18 architecture.By displaying all three metrics on the same graph, the plot helps us observe whether increasing model depth is associated with improved performance on the real-vs-AI image classification task.

## 8. Discussion and Interpretation

```markdown
## Contribution Metadata
Author(s):  Emily Jenks
Section:  Discussion and Interpretation
Method: [model_comparison, performance_analysis, depth_vs_accuracy, transfer_learning, overfitting, generalization]
SHORT Description of contribution:
In this section, I interpreted the final model results by comparing the performance of different
architectures on the test dataset and analyzing how model depth impacted accuracy. I examined trends
shown in the comparison plots, highlighting how deeper models such as ResNet18 outperformed shallow CNNs
in capturing complex image features. Additionally, I discussed the risk of overfitting and the limitations of
drawing strong conclusions about generalization from the test results alone.


Difficulty: BASIC
```


The "Performance Trend as Model Depth Increases" plot shows a consistent improvement across all metrics as we move from CNN-Small to CNN-Medium to ResNet18. CNN-Small has limited capacity (2 blocks, apprx. 20K parameters), so it mainly captures simple features like edges and textures. It struggles to detect more subtle artifacts that reveal AI-generated images, such as unnatural textures or inconsistencies in structure. When moving to CNN-Medium, the additional convolutional block increases the model’s capacity, allowing it to learn more complex patterns. This leads to a noticeable improvement in performance, including an apprx. 9 point accuracy gain and fewer false negatives.

The jump to ResNet18 (99.27% accuracy, AUC 0.9997) is much larger and cannot be explained by depth alone. Two main factors explain this. First, residual connections allow information and gradients to pass more effectively through the network, enabling deeper and more meaningful feature learning. This creates a hierarchy where early layers detect basic textures, and deeper layers capture more abstract patterns. Second, ResNet18 benefits from transfer learning using ImageNet, meaning it already has a strong understanding of general visual features. Instead of learning from scratch, it adapts this knowledge to the deepfake detection task, which is especially helpful given the limited dataset.

Despite its strong performance, ResNet18’s near-perfect results should be interpreted carefully. The test data comes from the same AI generators used in training, so the model may not generalize well to unseen sources like newer GANs or diffusion models. In contrast, the simpler CNNs may actually generalize better in some cases because they focus on more basic, broadly applicable features rather than highly specific patterns.

Looking at both the bar chart and the performance trend plot provides a clearer understanding of the results. The trend plot shows the improvement from CNN-Small depth-2 to CNN-Medium depth-3 is meaningful but incremental, while the jump to ResNet18 is near-vertical. This suggests that the major performance gain is not just from adding depth, but from the combination of residual architecture and transfer learning. Simply adding more layers to a shallow CNN would not achieve the same results.


## 9. Limitations and Fairness

```markdown
## Contribution Metadata
Author(s):  Chiara Bosi
Section:  
Method: [limited_dataset_size, bias, fairness]
SHORT Description of contribution:
In this section, I discuss about the main limitations, fairness considerations, and generalization challenges associated with the considered models, including dataset bias, explainability limitations and robustness concerns related to evolving generative AI models.

Difficulty: BASIC
```

Despite achieving strong classification performance, the project presents several limitations that should be considered when interpreting the results.

One important limitation concerns the dataset size and composition. Although transfer learning helps mitigate limited data availability, the classifier may still learn dataset-specific correlations that do not generalize perfectly to unseen real-world images. In particular, subtle biases related to image quality, compression artifacts, lighting conditions, or background regularities could unintentionally influence the predictions.

Another important challenge is the rapid evolution of generative AI models. Modern image synthesis techniques continuously improve the realism of generated images, meaning that the visual artifacts currently exploited by the classifier may become less detectable over time. Consequently, a model trained on today's synthetic images may not generalize equally well to future generations of AI-generated content.

Fairness is also an important consideration in image classification systems. The dataset may contain imbalances related to demographic representation, facial characteristics, skin tones, or image acquisition conditions. Such imbalances could affect the learned visual representations and potentially introduce unintended biases into the model predictions.

Additionally, explainability methods such as LIME and Integrated Gradients provide only approximate interpretations of the model behavior. While these approaches help identify image regions that contribute to the predictions, they should not be interpreted as exact representations of the internal reasoning process of the neural network.

Future work could explore:
- larger and more diverse datasets;
- evaluation across different generative models;
- bias analysis across demographic groups;
- multimodal approaches combining image analysis with metadata


## 10. Reproducibility


```markdown
## Contribution Metadata
Author(s):  Vatsalya Arora
Section:  Reproducibility
Method: [random_seeding, hardware_determinism, state_serialization, runtime_consistency, environment_dependency]
SHORT Description of contribution:
Reproducibility in machine learning means providing everything another person needs—code, package versions, exact random seeds,
and specific data splits—to independently replicate your precise experimental results.

In a group project or academic submission, it ensures that your instructor or teammates can run your code on their own machines
and get the exact same model weights, performance metrics, and charts with minimal effort.

Difficulty: INTERMEDIATE
```

To reproduce the experimental results of this project with minimal effort, please follow the structured environment configuration, dependency checklist, and execution order detailed below.

Runtime Considerations & Acceleration
1. Hardware Accelerator: This notebook is optimized for Google Colab. Because training deep models like ResNet18 and multiple shallow CNN architectures from scratch is computationally expensive, GPU acceleration must be enabled before running the code (Runtime -> Change runtime type -> T4 GPU or higher).

2. Execution Bypass (Pre-saved Weights): To make evaluation completely effortless and save time, training is set to optional. If the train flag in the notebook configuration is set to False, the script will automatically bypass the 40+ minute training loops. Instead, it will use gdown to securely fetch our pre-saved model weight archives (.pth bundles) directly from Google Drive, load them onto the respective network architectures in seconds, and jump straight to generating the final test evaluations, dataframes, and confusion matrices.

Execution Order
1. To guarantee that data splits remain identical and variables do not conflict, execute the notebook cells strictly in the following sequential order:

2. Environment Setup & External Installs: Run the cell-level pip commands (torchinfo and captum).

3. Library Imports & Global Seeding: Initialize all core modules and lock down the global seed environment variable (SEED = 42).

4. Data Mounting & Loading: Authenticate Google Drive or fetch the image source directories.

5. Train/Val/Test Splitting: Execute the split matrices (or load the fixed split CSV records to prevent train-test data leakage).

6. Model Initialization: Build the empty structural shells for the Shallow CNNs and ResNet18.

7. Execution Branching (Train vs. Load): Set your execution flag. If training from scratch, run the optimization blocks. If evaluating, execute the web-download links to extract the model weights.

8. Testing & Interpretation: Execute the evaluation loop to compute performance tables, plot confusion matrices, and render XAI (LIME / Integrated Gradients) image attributions.

External Files Needed
1. Image Dataset Repository: The source image directory structured under Data.

2. Pre-saved Model Weights (If bypassing training): Accessible via direct Google Drive share streams, which are fetched natively inside the code using fixed file IDs.

In [ ]:
libraries_to_check = [
    "torch",
    "torchvision",
    "torchinfo",
    "captum",
    "pandas",
    "numpy",
    "scikit-learn",  # package name for sklearn
    "scikit-image",  # package name for skimage
    "pillow",        # package name for PIL
    "matplotlib",
    "requests",
    "gdown"
]

print("===== RUNTIME ENVIRONMENT INFORMATION =====")
print(f"Python Core Version: {sys.version.split()[0]}")
print("-" * 43)

for lib in libraries_to_check:
    try:
        # Check standard metadata registry first
        version = importlib.metadata.version(lib)
        print(f"{lib:<25} : {version}")
    except importlib.metadata.PackageNotFoundError:
        try:
            # Fallback for internal or alias namings
            mod = sys.modules.get(lib) or __import__(lib)
            if hasattr(mod, "__version__"):
                print(f"{lib:<25} : {mod.__version__}")
            elif hasattr(mod, "version"):
                print(f"{lib:<25} : {mod.version}")
            else:
                print(f"{lib:<25} : Version attribute hidden or built-in")
        except Exception:
            print(f"{lib:<25} : Version not found (Check installation)")

print("-" * 43)

## 11. Final Conclusion

This project focused on classifying images as real or AI-generated, a relevant problem due to the growing realism of deepfakes and synthetic media.
We compared two shallow CNN baselines with a pretrained ResNet18 model to study whether increasing depth improves performance.
Random search was used to select the best ResNet18 hyperparameters and transfer learning strategy.
The results showed a clear performance improvement from `CNN-Small` to `CNN-Medium` and finally to ResNet18.
This suggests that deeper pretrained models can capture more complex visual patterns for real-vs-AI classification.
Explainability methods were also applied to make individual predictions more interpretable by highlighting influential image regions.
A limitation is that the model may still depend on dataset-specific artifacts and may not generalize to all unseen generators.
Future work should test larger, more diverse datasets and improve robustness against newer AI-generated images.


## 12. Self-Evaluation (Optional but Recommended)

Use the rubric below to self-assess your work before submission.

| Criterion | Self-score (0–4) | Short justification |
|---|---:|---|
| Problem definition and motivation |4 | The problem is clearly defined as real-vs-AI image classification and motivated by the relevance of deepfake and synthetic media detection. |
| Data handling | 3 |  The dataset is loaded, labelled, split, and transformed appropriately, although further dataset analysis could have been added.|
| Baseline design | 3 | Two shallow CNN baselines were implemented to provide a controlled comparison, however a more detailed analysis of the effect of increasing depth would have been more complete with more intermediate models, instead of just two and then the proposed one. |
| Model implementation | 4 | The project includes both custom CNN models and a pretrained ResNet18 model with transfer learning whose pipelines are correct and every choice justified. |
| Evaluation methodology | 4 | Models are evaluated using accuracy, AUC-ROC, and F1-score, giving a more complete view than a single metric. |
| Experimental results | 3 | Final results are summarized using comparison tables and plots, although more plots and a more comprehensive analysis would have been better. |
| Discussion and interpretation |  3| The results are interpreted in relation to model depth and transfer learning, although a deeper analysis could further improve the discussion. |
| Limitations  | 3 | The project acknowledges several limitations, but many more could be analyzed |
| Reproducibility  | 4 |  Seeds, saved model weights, saved search results, and clear code structure support reproducibility and reuse of the experiment.|

## Submission Checklist

Before submitting, verify that:
- [ ] all required sections are filled
- [ ] the notebook runs from top to bottom
- [ ] figures and tables are visible
- [ ] baseline and proposed model are both evaluated
- [ ] claims are supported by evidence
- [ ] AI usage is declared
- [ ] file paths are correct
- [ ] the notebook is clean and readable